(LaTeX command is defined in this cell.)
$\newcommand{\bm}[1]{\boldsymbol{#1}}$

# $\mathsf{SEC}$ Privacy Parameter Selection

This notebook selects concrete parameters for the privacy of $\mathsf{SEC}$ as a standalone secret sharing scheme (Theorem SEC-HCLLR-multi in the SEC paper), for the query budgets $Q \in \{0, 2^{64}\}$. It is adapted from `LaTSEC-Parameter-Selection-v2.2-fixedtailbound.ipynb`.

The lattice-estimator preliminaries are kept unchanged from the $\mathsf{LaTSEC}$ notebook for documentation, including the $\mathsf{MSIS}$ conversion and wrappers. The $\mathsf{SEC}$ privacy search itself uses only $\mathsf{MLWE}$, so from "Input Preparations" onwards all $\mathsf{MSIS}$ and abort-key computations are removed.

This copy fixes the hard-core modulus size at $\lceil\lg q\rceil = 32$, for applications of $\mathsf{SEC}$ other than threshold signatures. The modulus size is selected by `LG_Q_SOURCE` in "Input Preparations"; setting `LG_Q_SOURCE = 'LaTSEC'` reproduces `SEC-Parameter-Selection-v1.ipynb`. The constraints on the widths do not involve $q$, so $q$ affects the result only through the retained $\ell$.

# Imports

In [1]:
from estimator import *
from estimator.reduction import RC
from sage.all import oo, log

# Lattice Estimator Information

In [2]:
import os, subprocess, estimator
root = os.path.dirname(os.path.dirname(os.path.abspath(estimator.__file__)))
print("estimator imported from:", root)
if os.path.isdir(os.path.join(root, ".git")):
    print("commit:", subprocess.check_output(["git", "-C", root, "rev-parse", "HEAD"], text=True).strip())
    print("date:  ", subprocess.check_output(["git", "-C", root, "log", "-1", "--format=%cd"], text=True).strip())
else:
    print("not a git checkout (probably unpacked from a .zip)")

estimator imported from: /home/arzaki/lattice-estimator
commit: 27a581bb8e9d49f5e9e2db315bd48ac769d5f5f5
date:   Sat Jun 6 11:26:16 2026 +0100


In [3]:
print(subprocess.check_output(["git", "-C", root, "status", "--porcelain", "--untracked-files=no"], text=True) or "clean: no tracked files modified")

clean: no tracked files modified


# Lattice Estimator Preliminaries

## Parameter Conversion

### $\mathsf{MLWE}$ to $\mathsf{LWE}$ Parameter Conversion

Let $n,m,q,d \in \mathbb{Z}^+$, $\mathcal{R}=\mathbb{Z}[X]/(X^d+1)$, and $\mathcal{R}_q=\mathbb{Z}_q[X]/(X^d+1)$. Let $\psi$ be a distribution over $\mathbb{Z}$, and by abuse of notation also write $\psi$ for the induced coefficient-wise distribution over $\mathcal{R}_q$ obtained by sampling each coefficient independently from $\psi$ and reducing modulo $q$. Write $\psi^{n+m}$ for the product distribution over $\mathcal{R}_q^{n+m}$. The (decisional) $\mathsf{MLWE}_{n,m,q,d,\psi}$ problem is to distinguish $(\bm{A},\begin{bmatrix}\bm{I}_n \ \bm{A}\end{bmatrix}\bm{s})$ from $(\bm{A},\bm{u})$, where $\bm{A} \stackrel{\$}{\gets} \mathcal{R}_q^{n \times m}$, $\bm{s} \gets \psi^{n + m}$, and $\bm{u} \stackrel{\$}{\gets} \mathcal{R}_q^n$. All arithmetic is performed over $\mathcal{R}_q$. The advantage of an adversary $\mathcal{A}$ against $\mathsf{MLWE}_{n,m,q,d,\psi}$ problem (for security parameter $\kappa$) is defined as $\mathsf{Adv}_\mathcal{A}^\mathsf{MLWE}(\kappa) := \left|\Pr[1\gets \mathcal{A}(\bm{A},\begin{bmatrix}\bm{I}_n \ \bm{A}\end{bmatrix}\bm{s})] - \Pr[1 \gets \mathcal{A}(\bm{A},\bm{u})]\right|$, where the probabilities are taken over the stated sampling and the internal randomness of $\mathcal{A}$.

We recall that $\mathsf{MLWE}_{n,m,q,d,\sigma} \rightsquigarrow \mathsf{LWE}_{md,nd,q,\sigma}$, i.e., the lattice estimator consider the LWE of the form $\mathsf{LWE}_{N_\text{LWE},M_\text{LWE},q,\chi_{\bm{s}},\chi_{\bm{e}}}$.

In [4]:
def MLWE_to_LWE_params(n_rows, m_cols, q, d, sigma, tag=None):
    """
    Convert MLWE_{n,m,q,d,sigma} into coefficient-flattened LWE parameters accepted by lattice estimator.

    Conversion:
        MLWE_{n,m,q,d,sigma} -> LWE_{m*d,n*d,q,sigma}

    Assumptions:
        A in R_q^{n x m}
        b = A*z + e
        z <- D_sigma^m
        e <- D_sigma^n

    Returns:
        LWE.Parameters object.
    """

    N_lwe = m_cols * d      # ordinary LWE secret dimension
    M_lwe = n_rows * d      # ordinary LWE sample count

    psi = ND.DiscreteGaussian(sigma)

    if tag is None:
        tag = (f"MLWE n={n_rows}, m={m_cols}, "f"d={d}, q={q}, sigma={sigma}")

    return LWE.Parameters(
        n=N_lwe,
        q=q,
        Xs=psi,
        Xe=psi,
        m=M_lwe,
        tag=tag
    )

Testing for MLWE to LWE conversion.

In [5]:
n_MLWE = 2; m_MLWE = 4; q_MLWE = 3329; d_MLWE = 512; sigma_MLWE= 3
my_lwe_params = MLWE_to_LWE_params(n_MLWE, m_MLWE, q_MLWE, d_MLWE, sigma_MLWE)
print(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')


LWE estimate using ADPS16 cost model.

In [6]:
%%time
LWE.estimate(my_lwe_params, red_cost_model = RC.ADPS16)

Algorithm <estimator.lwe_bkw.CodedBKW object at 0x760b73bce4b0> on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with Calling ceil() on infinity or NaN
Algorithm functools.partial(<estimator.lwe_primal.PrimalHybrid object at 0x760b73bcdc70>, mitm=True, babai=True, red_cost_model=<estimator.reduction.ADPS16 object at 0x760b73b86cf0>, red_shape_model=<function GSA at 0x760b73b96160>) on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with The lower bound exceeds the upper bound.
bdd                  :: rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd
CPU times: user 21.1 s, sys: 69.8 ms, total: 21.1 s
Wall time: 21.2 s


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'usvp': rop: ≈2^inf, tag: usvp,
 'bdd': rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd,
 'bdd_hybrid': rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, ζ: 0, |S|: 1, d: 3072, prob: 1.0, ↻: 1, tag: hybrid,
 'dual': rop: ≈2^inf, mem: ≈2^inf, m: ≈2^inf, β: 47, tag: dual,
 'dual_hybrid': rop: ≈2^inf, red: ≈2^inf, guess: ≈2^2782857.0, β: 1023, p: 2, ζ: 0, t: 0, β': 1023, N: ≈2^2782843.4, m: ≈2^11.0}

### $\mathsf{MSIS}$ to $\mathsf{SIS}$ Conversion

Let $n,m,q,d\in\mathbb{Z}^+$, $\mathcal{R}=\mathbb{Z}[X]/(X^d+1)$, and $\mathcal{R}_q=\mathbb{Z}_q[X]/(X^d+1)$. Let $B_\mathsf{MSIS}\in\mathbb{R}^+$. The $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}}$ problem is: given $\bm{A}\stackrel{\$}{\gets}\mathcal{R}_q^{n\times m}$, find $\bm{x}\in\mathcal{R}_q^{n+m}$ such that
1. $0<\|\bm{x}\|_2\leq B_\mathsf{MSIS}$, and
2. $\begin{bmatrix}\bm{I}_n\ \bm{A}\end{bmatrix}\bm{x}=\bm{0}$ (over $\mathcal{R}_q$),

where $\|\cdot\|_2$ denotes the Euclidean norm of the coefficient embedding (using canonical representatives) and all arithmetic is performed over $\mathcal{R}_q$. The advantage of a PPT adversary $\mathcal{A}$ against $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}}$ (for security parameter $\kappa$) is $\mathsf{Adv}_\mathcal{A}^\mathsf{MSIS}(\kappa):= \Pr\big[\bm{A}\stackrel{\$}{\gets}\mathcal{R}_q^{n \times m}; \bm{x}\gets\mathcal{A}(\bm{A}): \left(0<\|\bm{x}\|_2\leq B_\mathsf{MSIS}\right)\land \left(\begin{bmatrix}\bm{I}_n\ \bm{A}\end{bmatrix}\bm{x}=\bm{0}\right)\big]$, where the probability is over $\bm{A}$ and the internal randomness of $\mathcal{A}$.

We recall that $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}} \rightsquigarrow \mathsf{SIS}_{nd,(n+m)d,q,B_\mathsf{MSIS}}$, i.e., the lattice estimator consider SIS of the form $\mathsf{SIS}_{N_\mathsf{MSIS},M_\mathsf{MSIS},q,B_\mathsf{MSIS}}$

In [7]:
def MSIS_to_SIS_params(n_rows, m_random_cols, q, d, B_MSIS, norm=2, tag=None):
    """
    Convert a systematic MSIS instance with matrix [I_n | A], where A is in R_q^{n x m}, into dimensional parameters for the ordinary SIS estimator.

    Dimension mapping:
        coefficient rows    = n*d
        coefficient columns = (n+m)*d
    This counts all coordinates of the systematic MSIS solution, but the lattice estimator does not explicitly model the identity block or the ring/module structure.

    The bound must satisfy B < (q-1)/2.  
    Under balanced representation every vector in Z_q^M has infinity norm at most (q-1)/2, so a bound at or above that makes the instance vacuous; 
    the estimator rejects it rather than returning a cost.  
    This is stricter than B < q, which only rules out the trivial solution q*e_1.

    Returns:
        SIS.Parameters object.
    """
    assert n_rows >= 1 and m_random_cols >= 1 and d >= 1
    assert 0 < B_MSIS < (q - 1)/2, (
        f"need 0 < B < (q-1)/2; the estimator rejects length_bound >= (q-1)/2 "
        f"as trivially easy (B = {B_MSIS}, q = {q})")
    N_sis = n_rows * d
    M_sis = (n_rows + m_random_cols) * d
    if tag is None:
        tag = (f"Systematic MSIS n={n_rows}, m={m_random_cols}, "
               f"d={d}, q={q}, B={B_MSIS}, norm={norm}")
    return SIS.Parameters(n=N_sis, q=q, length_bound=B_MSIS,
                          m=M_sis, norm=norm, tag=tag)

Testing for MSIS to SIS conversion.

In [8]:
n_MSIS = 2; m_MSIS = 4; q_MSIS = 3329; d_MSIS = 512; B_MSIS = 1000
my_sis_params = MSIS_to_SIS_params(n_MSIS,m_MSIS,q_MSIS,d_MSIS,B_MSIS)
print(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')


## Generic Security Analysis

Very roughly, after lattice reduction in dimension $D$, the first vector length is modeled as $\|\mathbf{b}_1\|_2 \approx \delta^{D} \mathrm{vol}(\Lambda)^{1/D}$ where $\mathrm{vol}(\Lambda)$ is the lattice volume/determinant. We have $\delta_0 \approx \left(\frac{\|\mathbf{b}_1\|_2}{\mathrm{vol}(\Lambda)^{1/D}}\right)^{1/D}$.

The value $\delta_0$ depends on the algorithm used:
1. LLL achieves $\delta_0 \approx 1.022$ in polynomial time, but the output quality is weak.
2. BKZ with block size $\beta$ achieves roughly $\delta_0 \approx \left(\frac{\beta}{2\pi e}(\pi \beta)^{1/\beta}\right)^{1/(2(\beta-1))}$.

Typically for:
1. $128$-bit security, we need $\delta_0 \approx 1.0045$,
2. $192$-bit security, we need $\delta_0 \approx 1.0029$,
3. $256$-bit security, we need $\delta_0 \approx 1.0023$.

### Helper function for Bit Security from RHF

In [9]:
from estimator.reduction import RC
from sage.all import oo, log

def log2_cost(x):
    """
    Return the base-2 logarithm of an estimator operation count.
    """
    if x == oo:
        return oo
    return float(log(x, 2))


def RHF_security_label(rhf):
    """
    Assign a heuristic security label based on RHF thresholds.
    """
    rhf = float(rhf)

    if rhf <= 1.0023:
        return "roughly >= 256-bit RHF level"
    elif rhf <= 1.0029:
        return "roughly >= 192-bit RHF level"
    elif rhf <= 1.0045:
        return "roughly >= 128-bit RHF level"
    else:
        return "below 128-bit RHF level"


### Summarizing the Security Estimate

In [10]:
def summarize_security_estimate(results):
    """
    Summarize an LWE or SIS lattice-estimator result.

    Extracts:
        - beta: BKZ block size
        - RHF: root-Hermite factor RC.delta(beta)
        - RHF_level: heuristic RHF security label
        - rop_bits: log2 of total attack cost
        - red_bits: log2 of lattice-reduction cost, if present
        - guess_bits: log2 of guessing cost, if present

    Parameters:
        results:
            Output of one of the following:

                LWE.estimate.rough(params)
                LWE.estimate(params)
                SIS.estimate.rough(params)
                SIS.estimate(params)

            For a single SIS.lattice(params) result, first wrap it as:

                {"lattice": SIS.lattice(params)}

    Returns:
        summary:
            Detailed information for each attack.

        RHF_data:
            BKZ block sizes, root-Hermite factors, and RHF labels.

        cheapest_attack:
            The attack with the smallest finite rop, or None when
            no finite rop is found.
    """
    summary = {}
    RHF_data = {}

    cheapest_attack_name = None
    cheapest_rop_bits = oo

    for attack_name, cost in results.items():
        entry = {}

        # Extract beta and compute the corresponding RHF.
        if "beta" in cost:
            beta = cost["beta"]

            if beta != oo:
                rhf = RC.delta(beta)
                rhf_level = RHF_security_label(rhf)

                entry["beta"] = beta
                entry["RHF"] = rhf
                entry["RHF_level"] = rhf_level

                RHF_data[attack_name] = {
                    "beta": beta,
                    "RHF": rhf,
                    "RHF_level": rhf_level
                }

        # Extract total attack cost.
        if "rop" in cost:
            rop_bits = log2_cost(cost["rop"])
            entry["rop_bits"] = rop_bits

            if rop_bits != oo and rop_bits < cheapest_rop_bits:
                cheapest_rop_bits = rop_bits
                cheapest_attack_name = attack_name

        # Extract lattice-reduction cost, if present.
        if "red" in cost:
            entry["red_bits"] = log2_cost(cost["red"])

        # Extract guessing cost, if present.
        if "guess" in cost:
            entry["guess_bits"] = log2_cost(cost["guess"])

        summary[attack_name] = entry

    if cheapest_attack_name is None:
        cheapest_attack = None
    else:
        cheapest_attack = {
            "attack": cheapest_attack_name,
            "rop_bits": cheapest_rop_bits
        }

    return summary, RHF_data, cheapest_attack

Printing the summary.

In [11]:
def print_security_summary(results, label="Security estimate"):
    """
    Print RHF and concrete bit-security information from an LWE or
    SIS lattice-estimator result.

    Parameters:
        results:
            Output of LWE.estimate.rough(...), LWE.estimate(...),
            SIS.estimate.rough(...), or SIS.estimate(...).

        label:
            Text used in the printed headings.

    Returns:
        summary, RHF_data, cheapest_attack
    """
    summary, RHF_data, cheapest_attack = (
        summarize_security_estimate(results)
    )

    print(f"\nRoot-Hermite factors from {label}:")

    if len(RHF_data) == 0:
        print("No finite beta found, so no RHF was computed.")
    else:
        for attack_name, data in RHF_data.items():
            print(
                f"{attack_name}: "
                f"beta = {data['beta']}, "
                f"RHF = {float(data['RHF'])}, "
                f"{data['RHF_level']}"
            )

    print(f"\nBit-security information from {label}:")

    if cheapest_attack is None:
        print("No finite rop found.")
    else:
        print(
            f"Cheapest finite attack: "
            f"{cheapest_attack['attack']}, "
            f"rop ≈ 2^{cheapest_attack['rop_bits']:.1f}"
        )

    return summary, RHF_data, cheapest_attack

### Some Testing

MLWE testing.

In [12]:
print(my_lwe_params)
lwe_test_result = LWE.estimate.rough(my_lwe_params)
lwe_test_summary, lwe_test_RHF, lwe_test_cheapest = print_security_summary(lwe_test_result,label="LWE.estimate.rough")

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')

Root-Hermite factors from LWE.estimate.rough:
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estimate.rough:
No finite rop found.


MSIS testing.

In [13]:
print(my_sis_params)
sis_test_result = SIS.estimate.rough(my_sis_params)
sis_test_summary, sis_test_RHF, sis_test_cheapest = print_security_summary(sis_test_result,label="SIS.estimate.rough")

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^460.8


/home/arzaki/lattice-estimator/estimator/util.py:387: UserWarning: red_shape_model is ignored for Euclidean (norm=2) SIS estimates; reduction shape simulators apply only to infinity-norm attacks.
  y = f(x)


In [14]:
sis_lattice_result = {"lattice": SIS.lattice(my_sis_params)}
sis_lattice_summary, sis_lattice_RHF, sis_lattice_cheapest = print_security_summary(sis_lattice_result,label="SIS.lattice")


Root-Hermite factors from SIS.lattice:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.lattice:
Cheapest finite attack: lattice, rop ≈ 2^472.5


## Wrapper Functions for LWE and SIS Analysis

### Wrapper functions for LWE analysis.

In [15]:
### ---- wrapper functions for LWE analysis --------------------------------

def run_LWE_rough_analysis(params, verbose=True):
    """
    Run the rough LWE security estimate and return the results.

    LWE.estimate.rough hard-codes the ADPS16 (core-SVP) cost model together with the ZGSA and costs only usvp and dual_hybrid, so it takes no cost model argument.  
    Because it runs a subset of the attacks, the reported Sec upper-bounds the full ADPS16 estimate.

    Parameters:
        params:
            An LWE.Parameters object, such as the output of MLWE_to_LWE_params(...).
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result:
            Raw output of LWE.estimate.rough(params).
        summary:
            Per-attack summary containing beta, RHF, rop_bits, red_bits, and guess_bits when available.
        RHF_data:
            Per-attack BKZ block sizes, RHFs, and RHF labels.
        Sec:
            lg of the smallest finite rop, or None if no attack returned a finite cost.  
            Sec is not reconstructed from beta: bdd and hybrid attacks add an svp or guess term to the reduction cost, so a small beta need not mean a cheap attack.
            None means the instance was not modeled, which is not the same as hard; the caller decides how to treat it.
    """
    result = LWE.estimate.rough(params)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(result, label="LWE.estimate.rough")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec


def run_LWE_full_analysis(params, cost_model=RC.ADPS16, verbose=True, deny_list=("arora-gb", "bkw")):
    """
    Run the full LWE security estimate and return the results.

    Costs the whole battery (usvp, bdd, bdd_hybrid, dual, dual_hybrid, and others).  
    Individual algorithms may fail on a given instance; those are reported on stdout by the estimator and are absent from result, 
    so Sec is a minimum over the attacks that succeeded.

    Parameters:
        params:
            An LWE.Parameters object, such as the output of MLWE_to_LWE_params(...).
        cost_model:
            Lattice-reduction cost model.
            RC.ADPS16 is core-SVP; RC.MATZOV is the estimator default and runs roughly 30 bits higher at the block sizes of interest.
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
        deny_list:
            Algorithms excluded from the estimate.  
            BKW and Arora-Gb need far more samples than a module instance provides 
            -- on a 3072-sample instance BKW reports needing 2^798 
            -- so they never enter the minimum, but searching their parameter space costs minutes.
            Excluding them is standard for module-lattice schemes.
            Pass () to cost everything.
    Returns:
        result:
            Raw output of LWE.estimate(params, red_cost_model=cost_model,deny_list=deny_list).
        summary, RHF_data, Sec:
            As in run_LWE_rough_analysis.
    """
    result = LWE.estimate(params, red_cost_model=cost_model,deny_list=deny_list)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(result, label="LWE.estimate")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec

Some testing.

In [16]:
%%time
print(my_lwe_params)
lwe_rough_result, lwe_rough_summary, lwe_RHF_rough, lwe_cheapest_rough = run_LWE_rough_analysis(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')

Root-Hermite factors from LWE.estimate.rough:
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estimate.rough:
No finite rop found.
CPU times: user 47.6 ms, sys: 0 ns, total: 47.6 ms
Wall time: 46.8 ms


In [17]:
%%time
print(my_lwe_params)
lwe_full_result, lwe_full_summary, lwe_RHF_full, lwe_cheapest_full = run_LWE_full_analysis(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')
Algorithm functools.partial(<estimator.lwe_primal.PrimalHybrid object at 0x760b73bcdc70>, mitm=True, babai=True, red_cost_model=<estimator.reduction.ADPS16 object at 0x760b73b86cf0>, red_shape_model=<function GSA at 0x760b73b96160>) on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with The lower bound exceeds the upper bound.
bdd                  :: rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd

Root-Hermite factors from LWE.estimate:
bdd: beta = 1753, RHF = 1.0013239694459048, roughly >= 256-bit RHF level
bdd_hybrid: beta = 1753, RHF = 1.0013239694459048, roughly >= 256-bit RHF level
dual: beta = 47, RHF = 1.0122323027109652, below 128-bit RHF level
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estim

### Wrapper Functions for SIS Analysis

In [18]:
### ---- wrapper functions for SIS analysis --------------------------------

def run_SIS_rough_analysis(params, verbose=True):
    """
    Run the rough SIS security estimate and return the results.

    SIS.estimate.rough hard-codes the ADPS16 (core-SVP) cost model, so it takes no cost model argument.  
    There is no runtime advantage over the full estimate: SIS costs a single lattice attack either way.  
    Retained for symmetry with the LWE wrappers.

    Parameters:
        params:
            A SIS.Parameters object, such as the output of MSIS_to_SIS_params(...).
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result, summary, RHF_data, Sec — as in run_LWE_rough_analysis.
    """
    result = SIS.estimate.rough(params)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(
            result, label="SIS.estimate.rough")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec


def run_SIS_full_analysis(params, cost_model=RC.ADPS16, verbose=True):
    """
    Run the full SIS security estimate and return the results.

    Parameters:
        params:
            A SIS.Parameters object, such as the output of MSIS_to_SIS_params(...).
        cost_model:
            Lattice-reduction cost model.  RC.ADPS16 is core-SVP; RC.MATZOV is the estimator default.
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result, summary, RHF_data, Sec — as in run_LWE_rough_analysis.
    """
    result = SIS.estimate(params, red_cost_model=cost_model)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(
            result, label="SIS.estimate")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec

Some testing.

In [19]:
%%time
print(my_sis_params)
sis_rough_result, sis_rough_summary, sis_RHF_rough, sis_cheapest_rough = run_SIS_rough_analysis(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^460.8
CPU times: user 1.17 ms, sys: 17 μs, total: 1.18 ms
Wall time: 1.05 ms


In [20]:
%%time
print(my_sis_params)
sis_full_result, sis_full_summary, sis_RHF_full, sis_cheapest_full = run_SIS_full_analysis(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate:
Cheapest finite attack: lattice, rop ≈ 2^460.8
CPU times: user 1.32 ms, sys: 13 μs, total: 1.34 ms
Wall time: 1.33 ms


### Testing for Large Quantities

Testing for:
1. $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}}$,
2. $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}}$,
3. $\mathsf{MSIS}_{p,\ell'+1,q,d,B_\mathsf{MSIS}}$,

where $p = 2$, $\ell' = 2$, $q = 2^{64}$, $d = 512$, $\sigma_\mathrm{MQ} = 2^{5.134}$, $\sigma_\mathrm{AK} = 2^{3.006}$.

In [21]:
from sage.all import RR

# Common parameters
p = 2; ell_prime = 2; q = 2^64; d = 512

# MLWE Gaussian standard deviations
sigma_MQ = RR(2)^RR("5.134")
sigma_AK = RR(2)^RR("3.006")

# MSIS Euclidean norm bound
B_MSIS = RR(2)^RR("59.1849")

print("p =", p)
print("ell_prime =", ell_prime)
print("q =", q)
print("d =", d)
print("sigma_MQ =", sigma_MQ)
print("sigma_AK =", sigma_AK)
print("B_MSIS =", B_MSIS)

p = 2
ell_prime = 2
q = 18446744073709551616
d = 512
sigma_MQ = 35.1146220025358
sigma_AK = 8.03334034591178
B_MSIS = 6.55284945157509e17


In [22]:
my_lwe_params_MQ = MLWE_to_LWE_params(p,ell_prime,q,d,sigma_MQ)
print(my_lwe_params_MQ)
my_lwe_params_AK = MLWE_to_LWE_params(p,ell_prime,q,d,sigma_AK)
print(my_lwe_params_AK)
my_sis_params_BMSIS = MSIS_to_SIS_params(p,ell_prime+1,q,d,B_MSIS,norm=2)
print(my_sis_params_BMSIS)

LWEParameters(n=1024, q=18446744073709551616, Xs=D(σ=35.11), Xe=D(σ=35.11), m=1024, tag='MLWE n=2, m=2, d=512, q=18446744073709551616, sigma=35.1146220025358')
LWEParameters(n=1024, q=18446744073709551616, Xs=D(σ=8.03), Xe=D(σ=8.03), m=1024, tag='MLWE n=2, m=2, d=512, q=18446744073709551616, sigma=8.03334034591178')
SISParameters(n=1024, q=18446744073709551616, length_bound=6.55284945157509e17, m=2560, norm=2, tag='Systematic MSIS n=2, m=3, d=512, q=18446744073709551616, B=6.55284945157509e17, norm=2')


In [23]:
%%time
(lwe_MQ_rough_result,lwe_MQ_rough_summary,lwe_MQ_RHF_rough,lwe_MQ_cheapest_rough) = run_LWE_rough_analysis(my_lwe_params_MQ)

usvp                 :: rop: ≈2^31.5, red: ≈2^31.5, δ: 1.008910, β: 108, d: 2047, tag: usvp
dual_hybrid          :: rop: ≈2^33.5, red: ≈2^33.3, guess: ≈2^30.6, β: 114, p: 3, ζ: 0, t: 0, β': 114, N: 1044.748, m: 1024

Root-Hermite factors from LWE.estimate.rough:
usvp: beta = 108, RHF = 1.0089095803238675, below 128-bit RHF level
dual_hybrid: beta = 114, RHF = 1.008665241318469, below 128-bit RHF level

Bit-security information from LWE.estimate.rough:
Cheapest finite attack: usvp, rop ≈ 2^31.5
CPU times: user 303 ms, sys: 0 ns, total: 303 ms
Wall time: 302 ms


In [24]:
%%time
(lwe_AK_rough_result,lwe_AK_rough_summary,lwe_AK_RHF_rough,lwe_AK_cheapest_rough) = run_LWE_rough_analysis(my_lwe_params_AK)

usvp                 :: rop: ≈2^27.2, red: ≈2^27.2, δ: 1.009588, β: 93, d: 2025, tag: usvp
dual_hybrid          :: rop: ≈2^28.7, red: ≈2^28.3, guess: ≈2^26.5, β: 97, p: 3, ζ: 0, t: 0, β': 97, N: 1137.413, m: 1024

Root-Hermite factors from LWE.estimate.rough:
usvp: beta = 93, RHF = 1.0095878948759565, below 128-bit RHF level
dual_hybrid: beta = 97, RHF = 1.0093969875477766, below 128-bit RHF level

Bit-security information from LWE.estimate.rough:
Cheapest finite attack: usvp, rop ≈ 2^27.2
CPU times: user 522 ms, sys: 0 ns, total: 522 ms
Wall time: 522 ms


In [25]:
%%time
(sis_full_result,sis_full_summary,sis_RHF_full,sis_cheapest_full) = run_SIS_rough_analysis(my_sis_params_BMSIS)

lattice  :: rop: ≈2^28.9, red: ≈2^28.9, δ: 1.009304, β: 99, d: 2214, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 99, RHF = 1.0093043507947785, below 128-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^28.9
CPU times: user 2.42 ms, sys: 0 ns, total: 2.42 ms
Wall time: 2.42 ms


# Input Preparations

$\mathsf{SEC}$ privacy parameters input.

We consider the retained instances $(n,k,k_p,k_c) \in \{(40,20,5,26),(60,40,6,46)\}$, the target security levels $\kappa \in \{128,192,256\}$, the query budgets $Q \in \{0,2^{64}\}$, and the ring degree $d = 512$.

The values $\alpha_{k_p}$, $\|\mathbf{\hat{G}}\|_2$, $\beta_{k_p}$, and $\gamma_{k_p}$ use the full precision obtained by exhaustive enumeration in `G-Related-Bounds/MatrixResult-40-19-3.ipynb` and `MatrixResult-60-39-3.ipynb`.
The values $\alpha_{k_p}$ and $\|\mathbf{\hat{G}}\|_2$ enter the precision parameter $\Xi_\mathrm{MQ}$, while $\beta_{k_p}$ and $\gamma_{k_p}$ enter only the kernel-smoothing threshold.
Unlike the $\mathsf{LaTSEC}$ search, $(\beta_{\bm{\lambda}},\omega_{\bm{\lambda}})$, $\|\mathbf{G}\|_2$, and $\beta_\mathbf{Q}$ are not needed.

Privacy places no lower bound on the hard-core modulus $q$, so $b_q := \lceil \lg q \rceil$ is an input, selected by `LG_Q_SOURCE`:
1. `'LaTSEC'`: $b_q$ is taken from the $\mathsf{LaTSEC}$ parameter sets with the same $(n,\kappa)$, where it is fixed by $\mathsf{MSIS}$;
2. `'fixed'`: $b_q$ is `LG_Q_FIXED` for every $(n,\kappa)$, for applications other than threshold signatures. This notebook uses $b_q = 32$.

Since the estimated $\mathsf{MLWE}$ security does not increase with $q$, we also report the largest supported $b_q$ for each retained candidate.

For fixed $\ell$, the $\mathsf{MLWE}$ security is largest when the secret rank $\ell'$ is largest, and $p \geq \ell'$ gives $\ell' \leq \lfloor \ell/2 \rfloor$.
We therefore consider only even $\ell$ with $p = \ell' = \ell/2$.

In [26]:
### ---- real number precision and constants ------------------------------

PREC = RealField(200)
PI   = PREC.pi()          # 200-bit pi, not the symbolic constant
LN2  = PREC(2).log()

### ---- SEC privacy parameter search: inputs ------------------------------

d = 512

# Retained SEC instances, full precision from
# G-Related-Bounds/MatrixResult-{40-19-3,60-39-3}.ipynb.
#   alpha_kp  = min_{|V| = k_p} sigma_min(Ghat_V)          -> Xi_MQ
#   norm_Ghat = ||Ghat||_2                                  -> Xi_MQ, sigma_eff
#   beta_kp   = max_{1 <= |U| <= k_p} ||Ghat_U||_2          -> tau_ker
#   gamma_kp  = min_{1 <= |U| <= k_p} sigma_min(M_U)        -> tau_ker
#   beta_g    = ||g||_inf
GEOMETRY = [
    dict(n=40, k=20, k_p=5, k_c=26, c_exp=3, beta_g=1,
         alpha_kp=0.46425590033034103, norm_Ghat=4.4316793740317,
         beta_kp=2.9403658464661486,   gamma_kp=0.19253138628300923),
    dict(n=60, k=40, k_p=6, k_c=46, c_exp=3, beta_g=1,
         alpha_kp=0.5176380902050413,  norm_Ghat=3.93699368153548,
         beta_kp=2.893489471774352,    gamma_kp=0.15210842969369098),
]

KAPPAS   = [128, 192, 256]
QUERIES  = [0, 2^64]               # Q = 0: standalone sharing; Q = 2^64: signing-query budget
LIST_ELL = list(range(2, 26, 2))   # even ell only, p = ell' = ell/2

# Hard-core modulus exponent b_q = ceil(lg q).  Privacy does not pin q, so b_q is an input.
#   'LaTSEC': b_q from the LaTSEC parameter sets with the same (n, kappa), fixed by MSIS.
#   'fixed' : one b_q for every (n, kappa), for applications other than threshold signatures.
LG_Q_SOURCE = 'fixed'
LG_Q_FIXED  = 32
LG_Q_LATSEC = {(40, 128): 61, (40, 192): 63, (40, 256): 64,
               (60, 128): 64, (60, 192): 66, (60, 256): 66}

if LG_Q_SOURCE == 'LaTSEC':
    LG_Q_TARGET = dict(LG_Q_LATSEC)
elif LG_Q_SOURCE == 'fixed':
    LG_Q_TARGET = {(g['n'], kap): LG_Q_FIXED for g in GEOMETRY for kap in KAPPAS}
else:
    raise ValueError(f"LG_Q_SOURCE must be 'LaTSEC' or 'fixed', not {LG_Q_SOURCE!r}")

### ---- model and search configuration -----------------------------------

SVP_CLASSICAL    = 0.292     # core-SVP sieving exponent; screening only
NTT_REQUIRED     = True      # if True, require q = 1 mod 2d for the hard-core modulus
BQ_HEADROOM_MAX  = 128       # upper end of the bisection for the largest supported b_q
ESTIMATOR_COMMIT = "27a581bb8e9d49f5e9e2db315bd48ac769d5f5f5"

### ---- input validation --------------------------------------------------

assert d & (d - 1) == 0, "d must be a power of two for R = Z[X]/(X^d+1)"
assert all(ell >= 2 and ell % 2 == 0 for ell in LIST_ELL), "ell must be even (p = ell' = ell/2)"
assert 0 in QUERIES and all(Q >= 0 for Q in QUERIES)
assert set(LG_Q_TARGET) == {(g['n'], kap) for g in GEOMETRY for kap in KAPPAS}
assert all(BQ_HEADROOM_MAX > b for b in LG_Q_TARGET.values())
for g in GEOMETRY:
    assert 1 <= g['k_p'] < g['k_c'] <= g['n'], g['n']
    assert g['k_p'] <= g['k'] - 1, g['n']                  # Ghat_U can have full row rank
    assert g['beta_g'] >= 1 and g['c_exp'] >= 1, g['n']
    assert 0 < g['gamma_kp'] <= g['alpha_kp'], g['n']       # adding columns raises sigma_min
    assert g['alpha_kp'] <= g['beta_kp'] <= g['norm_Ghat'], g['n']

Derived quantities.

1. Challenge weight: $\omega_c := \min\{\omega \in [d]: 2^\omega\binom{d}{\omega} \geq 2^\kappa\}$, following Hermine (Section 7), which gives $\omega_c \in \{19,31,44\}$. Unlike $\mathsf{LaTSEC}$, there is no random-oracle term $Q_H$. The weight $\omega_c$ affects the widths only when $Q \geq 1$.
2. Error allocation: $\epsilon' = \epsilon := 2^{-(\kappa+3)}$ and $\kappa_\mathsf{MLWE} := \kappa + 1$, so that $2^{-\kappa_\mathsf{MLWE}} + 2\epsilon' + 2\epsilon = 2^{-\kappa}$.
3. Screening block size: $\beta_\mathsf{MLWE} := \lceil \kappa_\mathsf{MLWE}/0.292 \rceil$ (core-SVP, classical), used only by the root-Hermite-factor pre-screen.
4. Kernel-geometry factor: $\Gamma_{k_p} := \sqrt{1 + \min\{k_p,\beta_{k_p}^2\}/\gamma_{k_p}^2} \geq \gamma_{k_p}^{\ker}$, which depends only on the profile.

In [27]:
### ---- derived quantities ------------------------------------------------

def _omega_c(kap):
    r"""omega_c := min{omega : 2^omega * binomial(d, omega) >= 2^kappa}  (Hermine, Sec. 7)."""
    for w in range(1, d + 1):
        if 2^w * binomial(d, w) >= 2^kap:
            return w
    assert False, f"no omega_c found for kappa={kap}"

OMEGA_C = {kap: _omega_c(kap) for kap in KAPPAS}

# eps' = eps = 2^-(kappa+3) and kappa_MLWE = kappa + 1, so that
# 2^-kappa_MLWE + 2 eps' + 2 eps = 2^-(kappa+1) + 2^-(kappa+1) = 2^-kappa.
EPSILON    = {kap: 2^(-(kap + 3)) for kap in KAPPAS}
KAPPA_MLWE = {kap: kap + 1 for kap in KAPPAS}

# Screening block size (core-SVP, classical), for the RHF pre-screen only.
BETA_MLWE = {kap: ceil(KAPPA_MLWE[kap] / SVP_CLASSICAL) for kap in KAPPAS}

def Gamma_kp(g):
    r"""Gamma_kp := sqrt(1 + min{k_p, beta_kp^2} / gamma_kp^2) >= gamma_kp^ker."""
    k_p, b, gm = PREC(g['k_p']), PREC(g['beta_kp']), PREC(g['gamma_kp'])
    return (1 + min(k_p, b^2) / gm^2).sqrt()

GAMMA_KER = {g['n']: Gamma_kp(g) for g in GEOMETRY}

### ---- validation --------------------------------------------------------

assert OMEGA_C == {128: 19, 192: 31, 256: 44}, OMEGA_C
assert all(EPSILON[kap] in QQ for kap in KAPPAS), "EPSILON must stay exact"
for kap in KAPPAS:
    assert 2^(-KAPPA_MLWE[kap]) + 4*EPSILON[kap] == 2^(-kap), kap    # exact in QQ
for g in GEOMETRY:
    assert GAMMA_KER[g['n']] >= 1, g['n']

print(f"{'kappa':>5} {'omega_c':>8} {'lg eps':>7} {'k_MLWE':>7} {'beta_MLWE':>10}")
for kap in KAPPAS:
    print(f"{kap:>5} {OMEGA_C[kap]:>8} {-(kap+3):>7} {KAPPA_MLWE[kap]:>7} {BETA_MLWE[kap]:>10}")
print()
print(f"{'n':>3} {'k_p':>4} {'beta_kp_sq':>10} {'gamma_kp':>10} {'Gamma_kp':>10}")
for g in GEOMETRY:
    b2 = PREC(g['beta_kp'])**2
    print(f"{g['n']:>3} {g['k_p']:>4} {b2:>10.4f} {PREC(g['gamma_kp']):>10.6f} "
          f"{GAMMA_KER[g['n']]:>10.4f}")

kappa  omega_c  lg eps  k_MLWE  beta_MLWE
  128       19    -131     129        442
  192       31    -195     193        661
  256       44    -259     257        881

  n  k_p beta_kp_sq   gamma_kp   Gamma_kp
 40    5     8.6458   0.192531    11.6570
 60    6     8.3723   0.152108    16.1346


Shorthand for $\lg(x) = \log_2(x)$.

In [28]:
def lg(x):
    return PREC(x).log(2)

Precision check for real number.

In [29]:
def assert_prec(store, name, fields):
    r"""Every named field in every entry of `store` must live in PREC."""
    for key, entry in store.items():
        bad = {f: entry[f].parent() for f in fields
               if entry[f].parent() is not PREC}
        assert not bad, f"{name}{key}: not at PREC — {bad}"

# Computing the Smoothing Thresholds

We use the smoothing parameter $\eta_\epsilon(\mathbb{Z}^N)$ in the standard-deviation normalization of the paper, together with the approximation $\eta_\epsilon(\mathbb{Z}^N) \approx \sqrt{\frac{\ln(2N/\epsilon)}{2\pi^2}}$. Write $D := d\ell$.

**Posterior resampling.** We define $S(N,\epsilon) := \sqrt{2}\eta_\epsilon(\mathbb{Z}^N) \approx \sqrt{\frac{\ln(2N/\epsilon)}{\pi^2}}$. For $\epsilon = 2^{-(\kappa+3)}$, we obtain
\begin{align*}
    S_\mathrm{MQ} := \sqrt{2}\eta_\epsilon(\mathbb{Z}^{D}) \approx \sqrt{\frac{\ln(2D) + (\kappa+3)\ln 2}{\pi^2}}.
\end{align*}

**Kernel smoothing.** With $\Gamma_{k_p} := \sqrt{1+\min\{k_p,\beta_{k_p}^2\}/\gamma_{k_p}^2}$ and $\epsilon' = 2^{-(\kappa+3)}$, the computable kernel-smoothing condition reads $\sigma_\mathrm{eff} \geq \tau_\mathrm{ker}$, where
\begin{align*}
    \tau_\mathrm{ker} := \Gamma_{k_p}\sqrt{\frac{\ln\left(2(k-1)D\left(2+1/\epsilon'\right)\right)}{2\pi^2}}.
\end{align*}
Unlike $S_\mathrm{MQ}$, this threshold carries no factor $\sqrt{2}$, is taken in dimension $(k-1)D$, and is scaled by $\Gamma_{k_p}$. Since $k \geq 2$ and $2 + 1/\epsilon' > 1/\epsilon$, we always have $\tau_\mathrm{ker} \geq \Gamma_{k_p} S_\mathrm{MQ}/\sqrt{2}$.

In [30]:
def S(N, kappa):
    r"""sqrt(2) * eta_eps(Z^N) at eps = EPSILON[kappa], standard-deviation normalization.

    In the paper's normalization, eta_eps(Z^N) ~ sqrt(ln(2N/eps) / (2 pi^2)),
    which is the pi-normalized value sqrt(ln(2N/eps)/pi) divided by sqrt(2 pi). Hence
        S(N, eps) := sqrt(2) * eta_eps(Z^N) ~ sqrt(ln(2N/eps)) / pi.
    2*N/EPSILON[kappa] is an exact integer, so the logarithm is the only
    inexact step and it is taken at PREC.
    """
    assert N >= 1 and kappa in KAPPAS
    return (PREC(2*N / EPSILON[kappa])).log().sqrt() / PI

def S_MQ(kappa, d, ell):
    r"""sqrt(2) * eta_eps(Z^(d*ell)), standard-deviation normalization."""
    assert d >= 1 and ell >= 1
    return S(d*ell, kappa)

def tau_ker(kappa, g, d, ell):
    r"""tau_ker := Gamma_kp * sqrt(ln(2 (k-1) d ell (2 + 1/eps')) / (2 pi^2)),  eps' = EPSILON[kappa].

    Right-hand side of the computable kernel-smoothing condition sigma_eff >= tau_ker.
    No sqrt(2) factor, dimension (k-1) d ell, and the kernel-geometry factor Gamma_kp.
    2 (k-1) d ell (2 + 1/eps') is an exact integer, so the logarithm is the only
    inexact step and it is taken at PREC.
    """
    assert kappa in KAPPAS and g['k'] >= 2 and d >= 1 and ell >= 1
    arg = 2 * (g['k'] - 1) * d * ell * (2 + 1/EPSILON[kappa])
    assert arg in ZZ
    return GAMMA_KER[g['n']] * (PREC(arg).log() / (2 * PI^2)).sqrt()

assert S(512*8, 128).parent().precision() == 200

# Sanity check: the new value is the old pi-normalized one divided by sqrt(2 pi).
_N, _k = 512*8, 128
_old = (2 * (PREC(2*_N / EPSILON[_k])).log() / PI).sqrt()
assert abs(S(_N, _k) * (2*PI).sqrt() / _old - 1) < PREC(2)^(-190)

Printing the smoothing thresholds.

In [31]:
### ---- smoothing thresholds over the search grid -------------------------

SMOOTH = {}
for g in GEOMETRY:
    for kappa in KAPPAS:
        for ell in LIST_ELL:
            SMOOTH[(g['n'], kappa, ell)] = dict(
                N_MQ    = d*ell,
                N_ker   = (g['k'] - 1)*d*ell,
                S_MQ    = S_MQ(kappa, d, ell),
                tau_ker = tau_ker(kappa, g, d, ell),
            )

assert_prec(SMOOTH, 'SMOOTH', ('S_MQ', 'tau_ker'))

for key, v in SMOOTH.items():
    assert 1 < v['S_MQ'] < 100, key
    # no sqrt(2) and a larger dimension: tau_ker >= Gamma_kp * S_MQ / sqrt(2)
    assert v['tau_ker'] >= GAMMA_KER[key[0]] * v['S_MQ'] / PREC(2).sqrt(), key

print(f"{'n':>3} {'kappa':>6} {'ell':>4} {'S_MQ':>8} {'tau_ker':>9} {'lg S_MQ':>8} {'lg tau':>7}")
for g in GEOMETRY:
    for kappa in KAPPAS:
        for ell in (LIST_ELL[0], 12, LIST_ELL[-1]):
            v = SMOOTH[(g['n'], kappa, ell)]
            print(f"{g['n']:>3} {kappa:>6} {ell:>4} {v['S_MQ']:>8.4f} {v['tau_ker']:>9.4f} "
                  f"{lg(v['S_MQ']):>8.4f} {lg(v['tau_ker']):>7.4f}")

  n  kappa  ell     S_MQ   tau_ker  lg S_MQ  lg tau
 40    128    2   3.1580   26.4168   1.6590  4.7234
 40    128   12   3.1866   26.6492   1.6720  4.7360
 40    128   24   3.1976   26.7386   1.6770  4.7409
 40    192    2   3.8036   31.6739   1.9274  4.9852
 40    192   12   3.8274   31.8680   1.9364  4.9940
 40    192   24   3.8366   31.9428   1.9398  4.9974
 40    256    2   4.3546   36.1749   2.1225  5.1769
 40    256   12   4.3754   36.3450   2.1294  5.1837
 40    256   24   4.3834   36.4105   2.1320  5.1863
 60    128    2   3.1580   36.6932   1.6590  5.1974
 60    128   12   3.1866   37.0138   1.6720  5.2100
 60    128   24   3.1976   37.1371   1.6770  5.2148
 60    192    2   3.8036   43.9482   1.9274  5.4577
 60    192   12   3.8274   44.2162   1.9364  5.4665
 60    192   24   3.8366   44.3194   1.9398  5.4699
 60    256    2   4.3546   50.1646   2.1225  5.6486
 60    256   12   4.3754   50.3996   2.1294  5.6553
 60    256   24   4.3834   50.4902   2.1320  5.6579


# Coefficients and Constants in Gaussian Parameter Expression

For the multi-query privacy reduction, the uniform source width and the effective perturbation width are given by
\begin{align*}
    \frac{1}{\sigma_\mathrm{MQ}^2} &= 2\left(\frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2}\right), &
    \frac{1}{\sigma_\mathrm{eff}^2} &= \frac{1}{\sigma_\mathbf{R}^2} + \frac{e_\mathrm{ker}}{\sigma_\mathbf{r}^2},
\end{align*}
where $a_\mathrm{MQ} := \beta_\mathbf{g}^2 k_p/\alpha_{k_p}^2$, $b_\mathrm{MQ} := \beta_\mathbf{g}^2\omega_c^2 Q\left(\sqrt{n} + \sqrt{k_p}\|\mathbf{\hat{G}}\|_2/\alpha_{k_p}\right)^2$, and $e_\mathrm{ker} := \omega_c^2 Q\|\mathbf{\hat{G}}\|_2^2$.
The posterior-resampling condition is $\sigma_\mathrm{MQ} \geq S_\mathrm{MQ}$, and the kernel-smoothing condition is $\sigma_\mathrm{eff} \geq \tau_\mathrm{ker}$.

We write $A_\mathrm{MQ} := 1/(2S_\mathrm{MQ}^2)$ and $A_\mathrm{ker} := 1/\tau_\mathrm{ker}^2$, so that the two conditions become the linear constraints
\begin{align*}
    \text{(P)}\quad & \frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2} \leq A_\mathrm{MQ}, \\
    \text{(K)}\quad & \frac{1}{\sigma_\mathbf{R}^2} + \frac{e_\mathrm{ker}}{\sigma_\mathbf{r}^2} \leq A_\mathrm{ker}.
\end{align*}
When $Q = 0$, we have $b_\mathrm{MQ} = e_\mathrm{ker} = 0$, no leakage mask is sampled, and (K) reduces to $\sigma_\mathbf{R} \geq \tau_\mathrm{ker}$.
Constraint (K) replaces the two abort-key constraints of the $\mathsf{LaTSEC}$ search.

## $a_\mathrm{MQ}$, $b_\mathrm{MQ}$, and $e_\mathrm{ker}$

In [32]:
### ---- coefficients in the Gaussian-parameter expressions ----------------
# 1/sigma_MQ^2  = 2 (1/sigma_s^2 + a_MQ/sigma_R^2 + b_MQ/sigma_r^2)
# 1/sigma_eff^2 = 1/sigma_R^2 + e_ker/sigma_r^2
# All three are independent of ell; b_MQ and e_ker vanish when Q = 0.

def a_MQ(k_p, alpha_kp, beta_g):
    r"""a_MQ := beta_g^2 k_p / alpha_{k_p}^2."""
    assert k_p >= 1 and PREC(alpha_kp) > 0 and beta_g >= 1
    return PREC(beta_g)^2 * PREC(k_p) / PREC(alpha_kp)^2

def b_MQ(omega_c, Q, n, k_p, norm_Ghat, alpha_kp, beta_g):
    r"""b_MQ := beta_g^2 omega_c^2 Q (sqrt(n) + sqrt(k_p) ||Ghat||_2 / alpha_{k_p})^2;  zero when Q = 0."""
    assert omega_c >= 1 and Q >= 0 and n >= 1 and k_p >= 1 and beta_g >= 1
    assert PREC(norm_Ghat) > 0 and PREC(alpha_kp) > 0
    inner = PREC(n).sqrt() + PREC(k_p).sqrt() * PREC(norm_Ghat) / PREC(alpha_kp)
    return PREC(beta_g)^2 * (omega_c^2 * Q) * inner^2

def e_ker(omega_c, Q, norm_Ghat):
    r"""e_ker := omega_c^2 Q ||Ghat||_2^2;  zero when Q = 0."""
    assert omega_c >= 1 and Q >= 0 and PREC(norm_Ghat) > 0
    return (omega_c^2 * Q) * PREC(norm_Ghat)^2

## $A_\mathrm{MQ}$ and $A_\mathrm{ker}$

In [33]:
def A_MQ(kappa, d, ell):
    r"""A_MQ := 1/(2 S_MQ^2) -- budget of the posterior-resampling constraint (P)."""
    return 1 / (2 * S_MQ(kappa, d, ell)^2)

def A_ker(kappa, g, d, ell):
    r"""A_ker := 1/tau_ker^2 -- budget of the kernel-smoothing constraint (K)."""
    return 1 / tau_ker(kappa, g, d, ell)^2

## Deriving the Tuple $(a_\mathrm{MQ},b_\mathrm{MQ},e_\mathrm{ker},A_\mathrm{MQ},A_\mathrm{ker})$

In [34]:
def coeffs_constants(kappa, g, Q, d, ell):
    r"""Constraint-side data at (kappa, profile g, Q, ell).  Returns (a_MQ, b_MQ, e_ker, A_MQ, A_ker).

    With x_s = 1/sigma_s^2, x_R = 1/sigma_R^2, x_r = 1/sigma_r^2:
        (P)  x_s + a_MQ x_R + b_MQ x_r <= A_MQ     <=>  sigma_MQ  >= S_MQ
        (K)        x_R      + e_ker x_r <= A_ker   <=>  sigma_eff >= tau_ker
    For Q = 0, b_MQ = e_ker = 0 and x_r is absent.
    omega_c is read from OMEGA_C, so its definition lives in exactly one place.
    """
    assert kappa in OMEGA_C and Q in QUERIES
    wc = OMEGA_C[kappa]
    return (a_MQ(g['k_p'], g['alpha_kp'], g['beta_g']),
            b_MQ(wc, Q, g['n'], g['k_p'], g['norm_Ghat'], g['alpha_kp'], g['beta_g']),
            e_ker(wc, Q, g['norm_Ghat']),
            A_MQ(kappa, d, ell),
            A_ker(kappa, g, d, ell))

# Quick look at one grid point: n = 40, kappa = 128, ell = 12.
for _Q in QUERIES:
    _a, _b, _e, _A, _AK = coeffs_constants(128, GEOMETRY[0], _Q, d, 12)
    _lab = "0" if _Q == 0 else "2**64"
    print(f"Q={_lab:>6}: a_MQ={float(_a):.4f}  b_MQ={float(_b):.4e}  e_ker={float(_e):.4e}  "
          f"A_MQ={float(_A):.6f}  A_ker={float(_AK):.6f}")

Q=     0: a_MQ=23.1982  b_MQ=0.0000e+00  e_ker=0.0000e+00  A_MQ=0.049240  A_ker=0.001408
Q= 2**64: a_MQ=23.1982  b_MQ=5.0984e+24  e_ker=1.3079e+23  A_MQ=0.049240  A_ker=0.001408


# Optimization Problem to Find $\sigma_\mathbf{s}$, $\sigma_\mathbf{R}$, and $\sigma_\mathbf{r}$

## Tail Factors, Bounds, and the Share-Size Objective

We use the following tail bounds:
1. $\beta_\mathbf{s} = \theta_\mathbf{s} \sigma_\mathbf{s}\sqrt{d\ell}$, where $\theta_\mathbf{s} \geq 1+\sqrt{\frac{2\kappa\ln 2}{d\ell}}$;
2. $\beta_\mathbf{R} = \theta_\mathbf{R}\sigma_\mathbf{R}$, where $\theta_\mathbf{R}\geq 1+\sqrt{2\kappa_\mathbf{R}\ln 2}$ and $\kappa_\mathbf{R}:= \kappa+\left\lceil\log_2((k-1)d\ell)\right\rceil$;
3. when $Q \geq 1$, $\beta_\mathbf{r} = \theta_\mathbf{r}\sigma_\mathbf{r}\sqrt{d\ell}$, where $\theta_\mathbf{r} \geq 1+\sqrt{\frac{2\kappa_\mathbf{r}\ln 2}{d\ell}}$ and $\kappa_\mathbf{r} := \kappa+\left\lceil\log_2(nQ)\right\rceil$.

These follow from the discrete Gaussian tail bound $\Pr[\|\mathbf{x}\|_2 > \theta\sigma\sqrt{N}] < C_\mathrm{tail}(\theta)^N$ for $\mathbf{x}\gets\mathcal{D}_{\mathbb{Z}^N,\sigma}$, where $C_\mathrm{tail}(\theta) := \theta\exp\left(\frac{1-\theta^2}{2}\right)$, together with $C_\mathrm{tail}(\theta)^N \leq 2^{-\kappa}$ whenever $\theta \geq 1+\sqrt{\frac{2\kappa\ln 2}{N}}$. The secret and the leakage masks use $N = d\ell$, and each entry of $\mathbf{R}$ uses $N = 1$. When $Q = 0$, no leakage mask is sampled, so $\theta_\mathbf{r}$ and $\beta_\mathbf{r}$ are undefined.

We define $\theta(\kappa,N) := 1+\sqrt{\frac{2\kappa\ln 2}{N}}$, so that $\theta_\mathbf{s} = \theta(\kappa, d\ell)$, $\theta_\mathbf{R} = \theta(\kappa_\mathbf{R}, 1)$, and $\theta_\mathbf{r} = \theta(\kappa_\mathbf{r}, d\ell)$ are the minimal admissible values.

The bound $\beta_\mathbf{s}$ controls $\|\mathbf{s}\|_2$ and is used as the bound on $\|\mathbf{s}\|_\infty$ in the share bound $\beta_\mathsf{sh} := \beta_\mathbf{g}\beta_\mathbf{s} + c_\mathrm{exp}\beta_\mathbf{R}$. Since the objective below uses $\sigma_\mathsf{sh}$, this choice affects only the reported value of $\beta_\mathsf{sh}$, not the widths.

In [35]:
### ---- Gaussian tail factors and bounds -----------------------------------
# theta(kappa_eff, N) := 1 + sqrt(2 kappa_eff ln2 / N)   (Lemma Discrete-Gaussian-Tail-IA).
# For x <- D_{Z^N, sigma}: Pr[||x||_2 > theta sigma sqrt(N)] < C_tail(theta)^N, with
# C_tail(theta) = theta exp((1 - theta^2)/2), and C_tail(theta)^N <= 2^(-kappa_eff) for this theta,
# since -ln C_tail(1+x) = x + x^2/2 - ln(1+x) >= x^2/2.
# The paper states theta_* >= this bound; we take the minimal admissible value.

def theta_tail(kappa_eff, N):
    r"""theta(kappa_eff, N) := 1 + sqrt(2 kappa_eff ln2 / N).

    N = d*ell for the secret and the leakage masks (l2 bound on a vector in Z^N);
    N = 1 for theta_R (bound on a single integer entry of R).
    """
    assert kappa_eff > 0 and N >= 1
    return 1 + (2 * PREC(kappa_eff) * LN2 / N).sqrt()

def C_tail(theta):
    r"""C_tail(theta) := theta * exp((1 - theta^2)/2), for theta > 1."""
    theta = PREC(theta)
    assert theta > 1
    return theta * ((1 - theta^2) / 2).exp()

def beta_s(sigma_s, kappa, d, ell):
    r"""beta_s = theta(kappa, d*ell) * sigma_s * sqrt(d*ell).

    An l2 bound on s in Z^(d ell), used as the l_inf bound in Lemma bound-for-si.
    """
    assert PREC(sigma_s) > 0 and kappa >= 1 and d >= 1 and ell >= 1
    N = d * ell
    return theta_tail(kappa, N) * PREC(sigma_s) * PREC(N).sqrt()

def beta_R(sigma_R, kappa, k, d, ell):
    r"""beta_R = theta(kappa_R, 1) * sigma_R, with kappa_R = kappa + ceil(lg((k-1) d ell)).

    No sqrt(d*ell) factor: beta_R bounds each integer entry of R in Z^{(k-1) x d ell}
    (tail bound with N = 1), and the union bound over the (k-1) d ell entries
    is absorbed into kappa_R.
    """
    assert PREC(sigma_R) > 0 and kappa >= 1 and k >= 2 and d >= 1 and ell >= 1
    kappa_R = kappa + Integer((k-1)*d*ell - 1).nbits()     # = kappa + ceil(lg((k-1) d ell))
    return theta_tail(kappa_R, 1) * PREC(sigma_R)

def beta_r(sigma_r, kappa, n, d, ell, Q):
    r"""beta_r = theta(kappa_r, d*ell) * sigma_r * sqrt(d*ell), with kappa_r = kappa + ceil(lg(n Q)).

    Defined only for Q >= 1: with Q = 0 no leakage mask is sampled.
    The union bound over at most n Q leakage masks is absorbed into kappa_r.
    """
    assert Q >= 1, "beta_r is undefined for Q = 0 (no leakage mask)"
    assert PREC(sigma_r) > 0 and kappa >= 1 and n >= 1 and d >= 1 and ell >= 1
    kappa_r = kappa + Integer(n*Q - 1).nbits()              # = kappa + ceil(lg(n Q))
    N = d * ell
    return theta_tail(kappa_r, N) * PREC(sigma_r) * PREC(N).sqrt()

# Sanity check: the chosen theta certifies C_tail(theta)^N <= 2^(-kappa_eff)
# over the dimensions and effective security levels used in the search.
for _kap in (128, 192, 256, 400):
    for _N in (1, 512*2, 512*24):
        _th = theta_tail(_kap, _N)
        assert _N * C_tail(_th).log() <= -_kap * LN2, (_kap, _N)

Quick look.

In [36]:
# Quick look at one grid point: n = 40, kappa = 128, ell = 12, Q = 2^64.
_g, _kap, _N, _Qmax = GEOMETRY[0], 128, d*12, max(QUERIES)
_kR = _kap + Integer((_g['k'] - 1)*_N - 1).nbits()
_kr = _kap + Integer(_g['n']*_Qmax - 1).nbits()
print(f"theta_s = {theta_tail(_kap, _N):.6f}   kappa_R = {_kR}  theta_R = {theta_tail(_kR, 1):.6f}   "
      f"kappa_r = {_kr}  theta_r = {theta_tail(_kr, _N):.6f}")

theta_s = 1.169944   kappa_R = 145  theta_R = 15.177894   kappa_r = 198  theta_r = 1.211366


**Objective.** Our objective is the size of one share. Each share coefficient has variance at most $\sigma_\mathsf{sh}^2 := \beta_\mathbf{g}^2\sigma_\mathbf{s}^2 + c_\mathrm{exp}\sigma_\mathbf{R}^2$, so one share has ideal entropy-coded size $D\left(\lg\sigma_\mathsf{sh} + \frac{1}{2}\lg(2\pi e)\right) \approx D(\lg\sigma_\mathsf{sh} + 2.047)$ bits, where $D := d\ell$. For fixed $\ell$, we therefore minimize $\lg\sigma_\mathsf{sh}$. With $x_\mathbf{s} := 1/\sigma_\mathbf{s}^2$, $x_\mathbf{R} := 1/\sigma_\mathbf{R}^2$, and $x_\mathbf{r} := 1/\sigma_\mathbf{r}^2$, we have $\sigma_\mathsf{sh}^2 = \frac{w_\mathbf{s}}{x_\mathbf{s}} + \frac{w_\mathbf{R}}{x_\mathbf{R}}$ with $w_\mathbf{s} := \beta_\mathbf{g}^2$ and $w_\mathbf{R} := c_\mathrm{exp}$.

The width $\sigma_\mathbf{r}$ enters only when $Q \geq 1$. In that case, the share size does not involve $x_\mathbf{r}$, and its minimum decreases towards the $Q = 0$ value as $\sigma_\mathbf{r} \to \infty$. We therefore also count one released leakage block $\mathbf{z}_{i,s} = \mathbf{L}_s\mathbf{s}_i + \mathbf{r}_{i,s}$, whose coefficients have width $\sqrt{\omega_c\sigma_\mathsf{sh}^2 + \sigma_\mathbf{r}^2} \approx \sigma_\mathbf{r}$, and minimize
\begin{align*}
    \lg\sigma_\mathsf{sh} + \lg\sigma_\mathbf{r} = \frac{1}{2}\lg\left(\frac{w_\mathbf{s}}{x_\mathbf{s}} + \frac{w_\mathbf{R}}{x_\mathbf{R}}\right) - \frac{1}{2}\lg x_\mathbf{r}.
\end{align*}
This choice is justified as follows.
1. A minimizer is Pareto-optimal for (share size, leakage-block size), and the weight $w = 1$ in $\lg\sigma_\mathsf{sh} + w\lg\sigma_\mathbf{r}$ counts one stored share and one released block equally.
2. When a single budget binds, it is split evenly between the leakage term and the sharing terms, and each object pays about half a bit per coefficient over its own optimum.
3. Any $w \in [1/2, 2]$ changes each overhead by at most about $0.3$ bit per coefficient. In contrast, minimizing $B_\mathbf{z} := \omega_c\sqrt{D}\beta_\mathsf{sh} + \beta_\mathbf{r}$, as in the $\mathsf{LaTSEC}$ search, inflates $\sigma_\mathbf{s}$ and $\sigma_\mathbf{R}$, because $\beta_\mathbf{r}$ dominates the sum.

We report $\beta_\mathsf{sh}$ and, when $Q \geq 1$, $B_\mathbf{z}$ as diagnostics.

In [37]:
### ---- share-size objective, size scale, and diagnostic bounds ------------
# Q = 0 :  minimize lg sigma_sh,             sigma_sh^2 = w_s/x_s + w_R/x_R
# Q >= 1:  minimize lg sigma_sh + lg sigma_r = (1/2) lg(w_s/x_s + w_R/x_R) - (1/2) lg x_r
# with w_s = beta_g^2 and w_R = c_exp.

LG_2PIE_HALF = lg(2*PI*PREC(1).exp()) / 2      # (1/2) lg(2 pi e) ~ 2.047

def objective_weights(g):
    r"""(w_s, w_R) with sigma_sh^2 = w_s/x_s + w_R/x_R, i.e. w_s = beta_g^2 and w_R = c_exp."""
    return PREC(g['beta_g'])^2, PREC(g['c_exp'])

def sigma_sh(sigma_s, sigma_R, g):
    r"""sigma_sh := sqrt(beta_g^2 sigma_s^2 + c_exp sigma_R^2), the size scale of one share coefficient."""
    w_s, w_R = objective_weights(g)
    return (w_s * PREC(sigma_s)^2 + w_R * PREC(sigma_R)^2).sqrt()

def objective_value(x_s, x_R, x_r, g):
    r"""lg sigma_sh               if x_r is None (Q = 0);
       lg sigma_sh + lg sigma_r  otherwise (Q >= 1), with sigma = 1/sqrt(x)."""
    w_s, w_R = objective_weights(g)
    val = lg(w_s / PREC(x_s) + w_R / PREC(x_R)) / 2
    return val if x_r is None else val - lg(PREC(x_r)) / 2

def beta_sh(b_s, b_R, g):
    r"""beta_sh := beta_g beta_s + c_exp beta_R  (Lemma bound-for-si)."""
    return PREC(g['beta_g']) * PREC(b_s) + PREC(g['c_exp']) * PREC(b_R)

def B_z(beta_sh_val, beta_r_val, omega_c, d, ell):
    r"""B_z := omega_c sqrt(d ell) beta_sh + beta_r, the bound on a released leakage block (Q >= 1).
    Diagnostic only; it is not the objective."""
    return PREC(omega_c) * PREC(d*ell).sqrt() * PREC(beta_sh_val) + PREC(beta_r_val)

def size_bits(sigma, D):
    r"""Ideal entropy-coded size D (lg sigma + (1/2) lg(2 pi e)) bits of D coefficients of width sigma."""
    assert PREC(sigma) > 0 and D >= 1
    return D * (lg(sigma) + LG_2PIE_HALF)

print(f"(1/2) lg(2 pi e) = {LG_2PIE_HALF:.4f}")

(1/2) lg(2 pi e) = 2.0471


## Preparing Coefficients and Constants

We minimize
\begin{align*}
    F(x_\mathbf{s},x_\mathbf{R},x_\mathbf{r}) :=
    \begin{cases}
        \frac{1}{2}\lg\left(\frac{w_\mathbf{s}}{x_\mathbf{s}} + \frac{w_\mathbf{R}}{x_\mathbf{R}}\right), & Q = 0,\\
        \frac{1}{2}\lg\left(\frac{w_\mathbf{s}}{x_\mathbf{s}} + \frac{w_\mathbf{R}}{x_\mathbf{R}}\right) - \frac{1}{2}\lg x_\mathbf{r}, & Q \geq 1,
    \end{cases}
\end{align*}
subject to
\begin{align*}
    \text{(P)}\quad & x_\mathbf{s} + a_\mathrm{MQ}x_\mathbf{R} + b_\mathrm{MQ}x_\mathbf{r} \leq A_\mathrm{MQ}, \\
    \text{(K)}\quad & x_\mathbf{R} + e_\mathrm{ker}x_\mathbf{r} \leq A_\mathrm{ker},
\end{align*}
where $w_\mathbf{s} := \beta_\mathbf{g}^2$, $w_\mathbf{R} := c_\mathrm{exp}$, $A_\mathrm{MQ}:= 1/(2S_\mathrm{MQ}^2)$, and $A_\mathrm{ker}:=1/\tau_\mathrm{ker}^2$. When $Q=0$, the variable $x_\mathbf{r}$ is absent.

When $Q \geq 1$, the constraints force $x_\mathbf{r} < \min\{A_\mathrm{MQ}/b_\mathrm{MQ}, A_\mathrm{ker}/e_\mathrm{ker}\}$, that is, $\sigma_\mathbf{r} > \sigma_\mathbf{r}^{\min} := \max\{\sqrt{b_\mathrm{MQ}/A_\mathrm{MQ}}, \sqrt{e_\mathrm{ker}/A_\mathrm{ker}}\}$, the smallest leakage width admitted by (P) and (K). We record $\sigma_\mathbf{r}^{\min}$ and which constraint determines it.

In [38]:
### ---- full coefficient set for the share-size minimization --------------

def objective_coeff_constants(kappa, g, Q, d, ell):
    r"""All coefficients for the width optimization at (kappa, profile g, Q, ell).

    Returns (a_MQ, b_MQ, e_ker, A_MQ, A_ker, w_s, w_R).

    Constraints, with x_s = 1/sigma_s^2, x_R = 1/sigma_R^2, x_r = 1/sigma_r^2:
        (P)  x_s + a_MQ x_R + b_MQ x_r <= A_MQ
        (K)        x_R      + e_ker x_r <= A_ker
    Objective:
        Q = 0 :  (1/2) lg(w_s/x_s + w_R/x_R)                    = lg sigma_sh
        Q >= 1:  (1/2) lg(w_s/x_s + w_R/x_R) - (1/2) lg x_r     = lg sigma_sh + lg sigma_r
    """
    return coeffs_constants(kappa, g, Q, d, ell) + objective_weights(g)

Generating the table.

In [39]:
### ---- assemble over the search grid -------------------------------------
# Keys are (n, kappa, Q, ell).  For Q >= 1 we also record the leakage-width floor
#     sigma_r_min := max{sqrt(b_MQ/A_MQ), sqrt(e_ker/A_ker)} = 1/sqrt(x_r_max),
# the smallest sigma_r admitted by (P) and (K), and which constraint sets it.

def Q_label(Q):
    r"""'0' or '2^m' for Q = 2^m (display only)."""
    if Q == 0:
        return "0"
    m = Integer(Q).nbits() - 1
    assert Q == 2^m, "Q must be 0 or a power of two"
    return "2^%d" % m

COEFF = {}
for g in GEOMETRY:
    for kappa in KAPPAS:
        for Q in QUERIES:
            for ell in LIST_ELL:
                (a_mq, b_mq, e_k, A_mq, A_k, w_s, w_R) = objective_coeff_constants(kappa, g, Q, d, ell)
                entry = dict(a_MQ=a_mq, b_MQ=b_mq, e_ker=e_k, A_MQ=A_mq, A_ker=A_k, w_s=w_s, w_R=w_R)
                if Q >= 1:
                    entry['x_r_max']     = min(A_mq/b_mq, A_k/e_k)
                    entry['sigma_r_min'] = 1 / entry['x_r_max'].sqrt()
                    entry['binding_r']   = 'MQ' if A_mq/b_mq <= A_k/e_k else 'ker'
                COEFF[(g['n'], kappa, Q, ell)] = entry

FIELDS = ('a_MQ', 'b_MQ', 'e_ker', 'A_MQ', 'A_ker', 'w_s', 'w_R')
assert_prec(COEFF, 'COEFF', FIELDS)

for key, c in COEFF.items():
    n, kappa, Q, ell = key
    assert all(c[f] > 0 for f in ('a_MQ', 'A_MQ', 'A_ker', 'w_s', 'w_R')), key
    if Q == 0:
        assert c['b_MQ'] == 0 and c['e_ker'] == 0 and 'x_r_max' not in c, key
    else:
        assert c['b_MQ'] > 0 and c['e_ker'] > 0 and c['x_r_max'] > 0, key
        assert c['x_r_max'].parent() is PREC and c['sigma_r_min'].parent() is PREC, key

def _fmt_lg(x, w=9):
    r"""lg x right-aligned in width w, or '--' when x = 0 (no leakage term)."""
    return "--".rjust(w) if x == 0 else ("%.4f" % float(lg(x))).rjust(w)

def show_coeffs(L=None):
    r"""Print the COEFF entries for ell in L (default: all of LIST_ELL)."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} | "
          f"{'lg a_MQ':>9} {'lg b_MQ':>9} {'lg e_ker':>9} {'lg A_MQ':>9} {'lg A_ker':>9} | "
          f"{'lg sr_min':>9} {'bind':>5}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for Q in QUERIES:
                for ell in L:
                    c = COEFF[(g['n'], kappa, Q, ell)]
                    sr   = _fmt_lg(c['sigma_r_min']) if Q >= 1 else "--".rjust(9)
                    bind = c['binding_r'] if Q >= 1 else "--"
                    print(f"{g['n']:>3} {kappa:>4} {Q_label(Q):>5} {ell:>4} | "
                          + " ".join(_fmt_lg(c[f]) for f in ('a_MQ', 'b_MQ', 'e_ker', 'A_MQ', 'A_ker'))
                          + f" | {sr} {bind:>5}")

Testing

In [40]:
show_coeffs([12])

  n  kap     Q  ell |   lg a_MQ   lg b_MQ  lg e_ker   lg A_MQ  lg A_ker | lg sr_min  bind
 40  128     0   12 |    4.5359        --        --   -4.3440   -9.4720 |        --    --
 40  128  2^64   12 |    4.5359   82.0763   76.7916   -4.3440   -9.4720 |   43.2102    MQ
 40  192     0   12 |    4.5359        --        --   -4.8727   -9.9881 |        --    --
 40  192  2^64   12 |    4.5359   83.4888   78.2041   -4.8727   -9.9881 |   44.1808    MQ
 40  256     0   12 |    4.5359        --        --   -5.2588  -10.3674 |        --    --
 40  256  2^64   12 |    4.5359   84.4993   79.2146   -5.2588  -10.3674 |   44.8791    MQ
 60  128     0   12 |    4.4849        --        --   -4.3440  -10.4200 |        --    --
 60  128  2^64   12 |    4.4849   81.9382   76.4500   -4.3440  -10.4200 |   43.4350   ker
 60  192     0   12 |    4.4849        --        --   -4.8727  -10.9330 |        --    --
 60  192  2^64   12 |    4.4849   83.3507   77.8626   -4.8727  -10.9330 |   44.3978   ker
 60  256  

## Optimization Model

We solve the problem in the variables $x := x_\mathbf{s}$, $y := x_\mathbf{R}$, and $z := x_\mathbf{r}$, with weights $W_x := w_\mathbf{s}$ and $W_y := w_\mathbf{R}$ and constraints
\begin{align*}
    x + ay + bz \leq P, \qquad y + cz \leq K,
\end{align*}
where $(a,b,c,P,K) = (a_\mathrm{MQ}, b_\mathrm{MQ}, e_\mathrm{ker}, A_\mathrm{MQ}, A_\mathrm{ker})$.

**Inner problem.** For fixed $z$ (or without $z$ when $Q = 0$), write $P_0 := P - bz$ and $K_0 := K - cz$, and minimize $W_x/x + W_y/y$ subject to $x + ay \leq P_0$ and $y \leq K_0$. The objective decreases in $x$, so the first constraint is active. The Lagrange condition gives $x/y = r := \sqrt{aW_x/W_y}$, hence the candidate $y^{(0)} = P_0/(a + r)$ and $x^{(0)} = r y^{(0)}$. If $y^{(0)} \leq K_0$, this is the solution (interior case); otherwise, $y = K_0$ and $x = P_0 - aK_0 > 0$ (clamp case). Since $\lg$ is increasing, the same point minimizes $\frac{1}{2}\lg(W_x/x + W_y/y)$.

**Case $Q = 0$.** The solution is the inner problem with $(P_0,K_0) = (P,K)$, so no numerical optimization is needed.

**Case $Q \geq 1$.** Feasibility requires $z < z_{\max} := \min\{P/b, K/c\}$. In the variables $(\ln x, \ln y, \ln z)$, the problem is a geometric program, so
\begin{align*}
    \Phi(u) := \min_{x,y}\left\{\tfrac{1}{2}\lg\left(\tfrac{W_x}{x}+\tfrac{W_y}{y}\right)\right\} - \tfrac{1}{2}\lg z, \qquad z = e^u,
\end{align*}
is convex in $u = \ln z$. We minimize $\Phi$ by golden-section search over $u \in (\ln z_{\max} - 64\ln 2, \ln z_{\max})$, that is, up to $64$ bits below $z_{\max}$, solving the inner problem at each step. Since $\Phi(u) \to \infty$ at both ends and the even split puts the minimizer about one bit below $z_{\max}$, the minimizer lies well inside the window.

In [41]:
### ---- constrained minimization of the share-size objective ---------------
# Variables x, y, z > 0 (inverse variances x_s, x_R, x_r).
#   (P)  x + a y + b z <= P
#   (K)      y   + c z <= K
# Q = 0 :  minimize (1/2) lg(Wx/x + Wy/y)                    (z absent)
# Q >= 1:  minimize (1/2) lg(Wx/x + Wy/y) - (1/2) lg z
# SEC mapping: (a, b, c, P, K) = (a_MQ, b_MQ, e_ker, A_MQ, A_ker), (Wx, Wy) = (w_s, w_R).

_INVPHI = (PREC(5).sqrt() - 1) / 2

def _golden_min(f, lo, hi, iters=300):
    r"""Golden-section minimum of a unimodal f on [lo, hi], at PREC.
    300 iterations shrink the bracket by 0.618^300 ~ 2^-208; the endpoints are never evaluated."""
    a, b = PREC(lo), PREC(hi)
    c, d_ = b - _INVPHI*(b - a), a + _INVPHI*(b - a)
    fc, fd = f(c), f(d_)
    for _ in range(iters):
        if fc < fd:
            b, d_, fd = d_, c, fc
            c = b - _INVPHI*(b - a); fc = f(c)
        else:
            a, c, fc = c, d_, fd
            d_ = a + _INVPHI*(b - a); fd = f(d_)
    return (a + b) / 2

def _inner(P0, K0, Wx, Wy, a):
    r"""argmin Wx/x + Wy/y  s.t.  x + a y <= P0,  y <= K0   (P0, K0 > 0).

    Interior (Lagrange): x/y = r := sqrt(a Wx / Wy), y = P0/(a + r), x = r y.
    If that y exceeds K0, clamp y = K0 and x = P0 - a K0 (> 0, since K0 < P0/(a + r) < P0/a).
    The first constraint is always active because the objective decreases in x.
    Returns (x, y, case) with case in {'interior', 'clamp'}.
    """
    r = (a * Wx / Wy).sqrt()
    y = P0 / (a + r)
    if y <= K0:
        return r * y, y, 'interior'
    return P0 - a * K0, K0, 'clamp'

def minimize_share_objective(Wx, Wy, a, b, c, P, K, with_z, u_window=None):
    r"""Minimize the share-size objective.

    with_z = False (Q = 0):  min (1/2) lg(Wx/x + Wy/y)            s.t. x + a y <= P,  y <= K.
    with_z = True  (Q >= 1): min (1/2) lg(Wx/x + Wy/y) - (1/2) lg z
                             s.t. x + a y + b z <= P,  y + c z <= K.
    For with_z, the value is convex in u = ln z (geometric program), so golden-section search over
    u in (ln z_max - u_window, ln z_max), z_max = min(P/b, K/c), finds the minimum to PREC;
    the inner problem at fixed z is _inner with budgets (P - b z, K - c z).
    The default window is 64 ln 2, i.e. 64 bits below z_max.

    Returns dict(x, y, z, value, case, slack_P, slack_K, phi_P, phi_K), where z is None for Q = 0
    and phi_P = b z / P, phi_K = c z / K are the budget fractions used by the leakage term.
    """
    Wx, Wy, a, P, K = (PREC(t) for t in (Wx, Wy, a, P, K))
    assert min(Wx, Wy, a, P, K) > 0, "all of Wx, Wy, a, P, K must be positive"

    def F(x, y, z):
        v = lg(Wx/x + Wy/y) / 2
        return v if z is None else v - lg(z) / 2

    if not with_z:
        x, y, case = _inner(P, K, Wx, Wy, a)
        z, b, c = None, PREC(0), PREC(0)
    else:
        b, c = PREC(b), PREC(c)
        assert b > 0 and c > 0, "b and c must be positive when Q >= 1"
        hi = min(P/b, K/c).log()
        lo = hi - (PREC(64)*LN2 if u_window is None else PREC(u_window))
        def Phi(u):
            z = u.exp()
            x, y, _ = _inner(P - b*z, K - c*z, Wx, Wy, a)
            return F(x, y, z)
        u = _golden_min(Phi, lo, hi)
        assert lo + (hi - lo)/1000 < u < hi, "optimum at the edge of the search window"
        z = u.exp()
        x, y, case = _inner(P - b*z, K - c*z, Wx, Wy, a)

    zz = PREC(0) if z is None else z
    return dict(x=x, y=y, z=z, value=F(x, y, z), case=case,
                slack_P=P - (x + a*y + b*zz), slack_K=K - (y + c*zz),
                phi_P=b*zz/P, phi_K=c*zz/K)

In [42]:
# Quick look at one grid point: n = 40, kappa = 128, ell = 12.
for _Q in QUERIES:
    _c = COEFF[(40, 128, _Q, 12)]
    _sol = minimize_share_objective(_c['w_s'], _c['w_R'], _c['a_MQ'], _c['b_MQ'], _c['e_ker'],
                                    _c['A_MQ'], _c['A_ker'], with_z=(_Q >= 1))
    _lr = "--" if _sol['z'] is None else "%.4f" % float(-lg(_sol['z'])/2)
    print(f"Q={Q_label(_Q):>5}: lg sig_s={float(-lg(_sol['x'])/2):.4f}  lg sig_R={float(-lg(_sol['y'])/2):.4f}  "
          f"lg sig_r={_lr:>7}  F={float(_sol['value']):.4f}  case={_sol['case']}  "
          f"phi_P={float(_sol['phi_P']):.3f}  phi_K={float(_sol['phi_K']):.3f}")

Q=    0: lg sig_s=2.9574  lg sig_R=4.7360  lg sig_r=     --  F=5.5486  case=clamp  phi_P=0.000  phi_K=0.000
Q= 2^64: lg sig_s=3.7298  lg sig_R=5.2219  lg sig_r=43.6462  F=49.6903  case=clamp  phi_P=0.546  phi_K=0.490


## Solving the Width Optimization

Converting $x$-values to $\sigma$-values. Since $x = \frac{1}{\sigma^2}$, then $\sigma = \frac{1}{\sqrt{x}}$.

In [43]:
def x_to_sigma(x):
    r"""sigma = 1/sqrt(x), inverting x = 1/sigma^2."""
    assert PREC(x) > 0
    return 1 / PREC(x).sqrt()

Optimizing the widths at one grid point $(n,\kappa,Q,\ell)$, with the coefficients read from `COEFF`. For $Q = 0$, no leakage mask is sampled, so $\sigma_\mathbf{r}$ and $x_\mathbf{r}$ are stored as `None`.

In [44]:
### ---- optimize the Gaussian widths at one grid point ---------------------

def optimize_widths(key):
    r"""Optimized widths at key = (n, kappa, Q, ell), with the coefficients read from COEFF.

    Returns sigma_s, sigma_R, sigma_r (None when Q = 0), the inverse variances,
    the objective value F, sigma_sh, the solver case, the slacks of (P) and (K),
    and the budget fractions phi_P, phi_K used by the leakage term.
    """
    n, kappa, Q, ell = key
    c = COEFF[key]
    g = next(G for G in GEOMETRY if G['n'] == n)
    sol = minimize_share_objective(c['w_s'], c['w_R'], c['a_MQ'], c['b_MQ'], c['e_ker'],
                                   c['A_MQ'], c['A_ker'], with_z=(Q >= 1))
    s_s, s_R = x_to_sigma(sol['x']), x_to_sigma(sol['y'])
    s_r = None if sol['z'] is None else x_to_sigma(sol['z'])
    return dict(sigma_s=s_s, sigma_R=s_R, sigma_r=s_r,
                x_s=sol['x'], x_R=sol['y'], x_r=sol['z'],
                F=sol['value'], sigma_sh=sigma_sh(s_s, s_R, g),
                case=sol['case'], slack_P=sol['slack_P'], slack_K=sol['slack_K'],
                phi_P=sol['phi_P'], phi_K=sol['phi_K'])

Constructing the table for diagnostic.

In [45]:
### ---- solve the width optimization over the search grid ------------------

from collections import Counter

WIDTHS = {key: optimize_widths(key) for key in COEFF}

assert_prec(WIDTHS, 'WIDTHS', ('sigma_s', 'sigma_R', 'x_s', 'x_R', 'F', 'sigma_sh',
                               'slack_P', 'slack_K', 'phi_P', 'phi_K'))

TOL = PREC(2)^(-150)
for key, w in WIDTHS.items():
    n, kappa, Q, ell = key
    c = COEFF[key]
    # (P) is active at the optimum, because the objective decreases in x_s.
    assert abs(w['slack_P']) <= TOL * c['A_MQ'], f"{key}: (P) not active"
    assert w['slack_K'] >= -TOL * c['A_ker'], f"{key}: (K) violated"
    if Q == 0:
        assert w['sigma_r'] is None and w['x_r'] is None, key
        assert abs(w['F'] - lg(w['sigma_sh'])) < TOL, f"{key}: F != lg sigma_sh"
    else:
        assert w['sigma_r'].parent() is PREC and w['x_r'].parent() is PREC, key
        assert w['sigma_r'] > c['sigma_r_min'], f"{key}: sigma_r below its floor"
        assert abs(w['F'] - lg(w['sigma_sh']) - lg(w['sigma_r'])) < TOL, \
            f"{key}: F != lg sigma_sh + lg sigma_r"

print("solver cases:", dict(Counter(w['case'] for w in WIDTHS.values())))

def widths_table(L=None):
    r"""Print the optimized widths for ell in L (default: all of LIST_ELL)."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} | {'lg sig_s':>9} {'lg sig_R':>9} {'lg sig_r':>9} | "
          f"{'lg sig_sh':>9} {'F':>8} | {'case':>8} {'phi_P':>6} {'phi_K':>6}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for Q in QUERIES:
                for ell in L:
                    w = WIDTHS[(g['n'], kappa, Q, ell)]
                    lr = ("--" if w['sigma_r'] is None
                          else "%.4f" % float(lg(w['sigma_r']))).rjust(9)
                    print(f"{g['n']:>3} {kappa:>4} {Q_label(Q):>5} {ell:>4} | "
                          f"{float(lg(w['sigma_s'])):>9.4f} {float(lg(w['sigma_R'])):>9.4f} {lr} | "
                          f"{float(lg(w['sigma_sh'])):>9.4f} {float(w['F']):>8.4f} | "
                          f"{w['case']:>8} {float(w['phi_P']):>6.3f} {float(w['phi_K']):>6.3f}")

solver cases: {'clamp': 144}


Some testing.

In [46]:
widths_table([12])

  n  kap     Q  ell |  lg sig_s  lg sig_R  lg sig_r | lg sig_sh        F |     case  phi_P  phi_K
 40  128     0   12 |    2.9574    4.7360        -- |    5.5486   5.5486 |    clamp  0.000  0.000
 40  128  2^64   12 |    3.7298    5.2219   43.6462 |    6.0442  49.6903 |    clamp  0.546  0.490
 40  192     0   12 |    3.2344    4.9940        -- |    5.8072   5.8072 |    clamp  0.000  0.000
 40  192  2^64   12 |    4.0378    5.4767   44.6138 |    6.3012  50.9150 |    clamp  0.549  0.488
 40  256     0   12 |    3.4344    5.1837        -- |    5.9971   5.9971 |    clamp  0.000  0.000
 40  256  2^64   12 |    4.2548    5.6644   45.3108 |    6.4901  51.8010 |    clamp  0.550  0.486
 60  128     0   12 |    2.4630    5.2100        -- |    6.0078   6.0078 |    clamp  0.000  0.000
 60  128  2^64   12 |    2.6704    5.7112   43.9338 |    6.5072  50.4410 |    clamp  0.333  0.501
 60  192     0   12 |    2.7313    5.4665        -- |    6.2644   6.2644 |    clamp  0.000  0.000
 60  192  2^64   12 

## Deriving $\sigma_\mathrm{MQ}$ and $\sigma_\mathrm{eff}$ and Some Checking

We recompute, from the optimized widths,
\begin{align*}
    \sigma_\mathrm{MQ} &:= \frac{1}{\sqrt{2\left(\frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2}\right)}}, &
    \sigma_\mathrm{eff} &:= \frac{1}{\sqrt{\frac{1}{\sigma_\mathbf{R}^2} + \frac{e_\mathrm{ker}}{\sigma_\mathbf{r}^2}}},
\end{align*}
where the $\sigma_\mathbf{r}$ terms are absent when $Q = 0$, together with the ratios $R_\mathrm{MQ} := \sigma_\mathrm{MQ}/S_\mathrm{MQ}$ and $R_\mathrm{ker} := \sigma_\mathrm{eff}/\tau_\mathrm{ker}$. Feasibility requires $R_\mathrm{MQ} \geq 1$ and $R_\mathrm{ker} \geq 1$.

Since (P) is active at the optimum, $R_\mathrm{MQ} = 1$, that is, $\sigma_\mathrm{MQ} = S_\mathrm{MQ}$, which depends only on $(\kappa,\ell)$. Consequently, the instance $\mathsf{MLWE}_{\ell/2,\ell/2,q,d,\sigma_\mathrm{MQ}}$ depends only on $(\kappa,\ell,q)$: the profile and the query budget affect the widths and the sizes, but not the choice of $\ell$ or $q$. Moreover, $\sigma_\mathrm{MQ} \leq \sigma_\mathbf{s}/\sqrt{2}$, so the empty-view instance $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathbf{s}}$ is implied in practice. Unlike the $\mathsf{LaTSEC}$ search, there is no abort-key width $\sigma_\mathrm{AK}$.

In [47]:
### ---- recompute and validate the source widths ---------------------------

def sigma_MQ(sigma_s, sigma_R, sigma_r, aM, bM):
    r"""sigma_MQ = 1/sqrt(2(1/sigma_s^2 + aM/sigma_R^2 + bM/sigma_r^2)).
    The last term is absent when sigma_r is None (Q = 0)."""
    assert PREC(sigma_s) > 0 and PREC(sigma_R) > 0
    inv2 = 1/PREC(sigma_s)^2 + PREC(aM)/PREC(sigma_R)^2
    if sigma_r is not None:
        assert PREC(sigma_r) > 0
        inv2 += PREC(bM)/PREC(sigma_r)^2
    return 1/(2*inv2).sqrt()

def sigma_eff(sigma_R, sigma_r, eK):
    r"""sigma_eff = 1/sqrt(1/sigma_R^2 + eK/sigma_r^2); equals sigma_R when sigma_r is None (Q = 0)."""
    assert PREC(sigma_R) > 0
    inv2 = 1/PREC(sigma_R)^2
    if sigma_r is not None:
        assert PREC(sigma_r) > 0
        inv2 += PREC(eK)/PREC(sigma_r)^2
    return 1/inv2.sqrt()

def R_MQ(sigma_s, sigma_R, sigma_r, aM, bM, kappa, d, ell):
    r"""R_MQ = sigma_MQ / S_MQ.  Feasibility requires R_MQ >= 1."""
    return sigma_MQ(sigma_s, sigma_R, sigma_r, aM, bM) / S_MQ(kappa, d, ell)

def R_ker(sigma_R, sigma_r, eK, kappa, g, d, ell):
    r"""R_ker = sigma_eff / tau_ker.  Feasibility requires R_ker >= 1."""
    return sigma_eff(sigma_R, sigma_r, eK) / tau_ker(kappa, g, d, ell)

Computing the source MLWE widths at one grid point.

In [48]:
### ---- source widths at one grid point ------------------------------------

def sigma_MLWE_MQ_eff(key):
    r"""sigma_MQ, sigma_eff, R_MQ, R_ker at key = (n, kappa, Q, ell).

    Reads the optimized widths from WIDTHS and the coefficients from COEFF.
    """
    n, kappa, Q, ell = key
    c, w = COEFF[key], WIDTHS[key]
    g = next(G for G in GEOMETRY if G['n'] == n)
    sig = (w['sigma_s'], w['sigma_R'], w['sigma_r'])
    return dict(
        sigma_MQ=sigma_MQ(*sig, c['a_MQ'], c['b_MQ']),
        sigma_eff=sigma_eff(w['sigma_R'], w['sigma_r'], c['e_ker']),
        R_MQ=R_MQ(*sig, c['a_MQ'], c['b_MQ'], kappa, d, ell),
        R_ker=R_ker(w['sigma_R'], w['sigma_r'], c['e_ker'], kappa, g, d, ell),
    )

Constructing the table.

In [49]:
### ---- source MLWE width and effective width over the search grid ---------

SIGMA_MLWE = {key: sigma_MLWE_MQ_eff(key) for key in COEFF}
assert_prec(SIGMA_MLWE, 'SIGMA_MLWE', ('sigma_MQ', 'sigma_eff', 'R_MQ', 'R_ker'))

TOL = PREC(2)^(-150)
for key, s in SIGMA_MLWE.items():
    w = WIDTHS[key]
    # (P) is active at the optimum, so sigma_MQ = S_MQ exactly (R_MQ = 1).
    assert abs(s['R_MQ'] - 1) < TOL, f"{key}: R_MQ = {s['R_MQ']}, expected 1"
    # (K) must hold, and it is active (R_ker = 1) exactly in the clamp case.
    assert s['R_ker'] >= 1 - TOL, f"{key}: R_ker = {s['R_ker']} < 1 (kernel smoothing fails)"
    assert (abs(s['R_ker'] - 1) < TOL) == (w['case'] == 'clamp'), key
    # Empty view: sigma_MQ <= sigma_s/sqrt(2), so MLWE with width sigma_s is implied.
    assert s['sigma_MQ'] <= w['sigma_s'] / PREC(2).sqrt(), key

# sigma_MQ = S_MQ(kappa, ell) depends neither on the profile n nor on Q.
for kappa in KAPPAS:
    for ell in LIST_ELL:
        vals = [SIGMA_MLWE[(g['n'], kappa, Q, ell)]['sigma_MQ'] for g in GEOMETRY for Q in QUERIES]
        assert max(vals) - min(vals) < TOL * max(vals), (kappa, ell)
print("sigma_MQ depends only on (kappa, ell): the MLWE instance is the same for both profiles and both Q.")

def sigma_MLWE_table(L=None):
    r"""Print the source widths and smoothing ratios for ell in L (default: all of LIST_ELL)."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} | "
          f"{'lg sig_MQ':>10} {'lg sig_eff':>10} {'lg S_MQ':>9} {'lg tau':>9} | "
          f"{'R_MQ':>7} {'R_ker':>7}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for Q in QUERIES:
                for ell in L:
                    key = (g['n'], kappa, Q, ell)
                    s, S_ = SIGMA_MLWE[key], SMOOTH[(g['n'], kappa, ell)]
                    print(f"{g['n']:>3} {kappa:>4} {Q_label(Q):>5} {ell:>4} | "
                          f"{float(lg(s['sigma_MQ'])):>10.4f} {float(lg(s['sigma_eff'])):>10.4f} "
                          f"{float(lg(S_['S_MQ'])):>9.4f} {float(lg(S_['tau_ker'])):>9.4f} | "
                          f"{float(s['R_MQ']):>7.4f} {float(s['R_ker']):>7.4f}")

sigma_MQ depends only on (kappa, ell): the MLWE instance is the same for both profiles and both Q.


Printing a table.

In [50]:
sigma_MLWE_table([12])

  n  kap     Q  ell |  lg sig_MQ lg sig_eff   lg S_MQ    lg tau |    R_MQ   R_ker
 40  128     0   12 |     1.6720     4.7360    1.6720    4.7360 |  1.0000  1.0000
 40  128  2^64   12 |     1.6720     4.7360    1.6720    4.7360 |  1.0000  1.0000
 40  192     0   12 |     1.9364     4.9940    1.9364    4.9940 |  1.0000  1.0000
 40  192  2^64   12 |     1.9364     4.9940    1.9364    4.9940 |  1.0000  1.0000
 40  256     0   12 |     2.1294     5.1837    2.1294    5.1837 |  1.0000  1.0000
 40  256  2^64   12 |     2.1294     5.1837    2.1294    5.1837 |  1.0000  1.0000
 60  128     0   12 |     1.6720     5.2100    1.6720    5.2100 |  1.0000  1.0000
 60  128  2^64   12 |     1.6720     5.2100    1.6720    5.2100 |  1.0000  1.0000
 60  192     0   12 |     1.9364     5.4665    1.9364    5.4665 |  1.0000  1.0000
 60  192  2^64   12 |     1.9364     5.4665    1.9364    5.4665 |  1.0000  1.0000
 60  256     0   12 |     2.1294     5.6553    2.1294    5.6553 |  1.0000  1.0000
 60  256  2^64  

# Computing the Share and Leakage Bounds

We compute the bounds of the Gaussian tail section at the optimized widths:
1. $\beta_\mathbf{s} = \theta_\mathbf{s}\sigma_\mathbf{s}\sqrt{d\ell}$, $\beta_\mathbf{R} = \theta_\mathbf{R}\sigma_\mathbf{R}$, and, when $Q \geq 1$, $\beta_\mathbf{r} = \theta_\mathbf{r}\sigma_\mathbf{r}\sqrt{d\ell}$, with the minimal admissible tail factors;
2. the share bound $\beta_\mathsf{sh} := \beta_\mathbf{g}\beta_\mathbf{s} + c_\mathrm{exp}\beta_\mathbf{R}$, so that $\max_{i\in[n]}\|\mathbf{s}_i\|_\infty \leq \beta_\mathsf{sh}$ except with probability at most $2\cdot 2^{-\kappa}$;
3. when $Q \geq 1$, the leakage-block bound $B_\mathbf{z} := \omega_c\sqrt{d\ell}\,\beta_\mathsf{sh} + \beta_\mathbf{r}$, which bounds $\|\mathbf{z}_{i,s}\|_2$ on the good-tail event.

These bounds are diagnostics and are not used by the objective. Unlike the $\mathsf{LaTSEC}$ search, there is no aggregate bound $B_2$, no $B_\mathsf{STMSIS}$, and no $B_\mathsf{MSIS}$. The table also shows the ideal entropy-coded size of one share, $d\ell\left(\lg\sigma_\mathsf{sh} + 2.047\right)$ bits.

Computing the bounds at one grid point.

In [51]:
### ---- share and leakage bounds at one grid point -------------------------

def norm_bounds(key):
    r"""beta_s, beta_R, beta_r, beta_sh, B_z at key = (n, kappa, Q, ell).

    beta_r and B_z are None for Q = 0, since no leakage mask is sampled.
    """
    n, kappa, Q, ell = key
    w = WIDTHS[key]
    g = next(G for G in GEOMETRY if G['n'] == n)
    b_s  = beta_s(w['sigma_s'], kappa, d, ell)
    b_R  = beta_R(w['sigma_R'], kappa, g['k'], d, ell)
    b_sh = beta_sh(b_s, b_R, g)
    if Q == 0:
        b_r, Bz = None, None
    else:
        b_r = beta_r(w['sigma_r'], kappa, n, d, ell, Q)
        Bz  = B_z(b_sh, b_r, OMEGA_C[kappa], d, ell)
    return dict(beta_s=b_s, beta_R=b_R, beta_r=b_r, beta_sh=b_sh, B_z=Bz)

Constructing the table.

In [52]:
### ---- share and leakage bounds over the search grid -----------------------

NORM_BOUNDS = {key: norm_bounds(key) for key in COEFF}
assert_prec(NORM_BOUNDS, 'NORM_BOUNDS', ('beta_s', 'beta_R', 'beta_sh'))

for key, b in NORM_BOUNDS.items():
    n, kappa, Q, ell = key
    if Q == 0:
        assert b['beta_r'] is None and b['B_z'] is None, key
    else:
        assert b['beta_r'].parent() is PREC and b['B_z'].parent() is PREC, key
        # beta_r dominates B_z: the leakage mask carries the sqrt(Q) factor.
        assert b['beta_r'] / b['B_z'] > PREC(1)/2, f"{key}: beta_r/B_z = {b['beta_r']/b['B_z']}"

def _fmt_lg_or_dash(x, w):
    r"""lg x right-aligned in width w, or '--' when x is None."""
    return ("--" if x is None else "%.4f" % float(lg(x))).rjust(w)

def norm_bounds_table(L=None):
    r"""Print the share and leakage bounds and the share size for ell in L."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} | "
          f"{'lg beta_s':>10} {'lg beta_R':>10} {'lg beta_r':>10} | "
          f"{'lg beta_sh':>10} {'lg B_z':>8} | {'share KiB':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for Q in QUERIES:
                for ell in L:
                    key = (g['n'], kappa, Q, ell)
                    b = NORM_BOUNDS[key]
                    share_KiB = size_bits(WIDTHS[key]['sigma_sh'], d*ell) / (8*1024)
                    print(f"{g['n']:>3} {kappa:>4} {Q_label(Q):>5} {ell:>4} | "
                          f"{_fmt_lg_or_dash(b['beta_s'], 10)} {_fmt_lg_or_dash(b['beta_R'], 10)} "
                          f"{_fmt_lg_or_dash(b['beta_r'], 10)} | "
                          f"{_fmt_lg_or_dash(b['beta_sh'], 10)} {_fmt_lg_or_dash(b['B_z'], 8)} | "
                          f"{float(share_KiB):>10.3f}")

Printing a table.

In [53]:
norm_bounds_table([12])

  n  kap     Q  ell |  lg beta_s  lg beta_R  lg beta_r | lg beta_sh   lg B_z |  share KiB
 40  128     0   12 |     9.4763     8.6599         -- |    10.9112       -- |      5.697
 40  128  2^64   12 |    10.2487     9.1458    50.2153 |    11.5098  50.2153 |      6.068
 40  192     0   12 |     9.7997     9.1657         -- |    11.3522       -- |      5.891
 40  192  2^64   12 |    10.6030     9.6484    51.2203 |    11.9523  51.2203 |      6.261
 40  256     0   12 |    10.0376     9.5380         -- |    11.6800       -- |      6.033
 40  256  2^64   12 |    10.8580    10.0187    51.9495 |    12.2785  51.9495 |      6.403
 60  128     0   12 |     8.9819     9.1385         -- |    11.1009       -- |      6.041
 60  128  2^64   12 |     9.1894     9.6397    50.5029 |    11.5396  50.5029 |      6.416
 60  192     0   12 |     9.2965     9.6414         -- |    11.5626       -- |      6.234
 60  192  2^64   12 |     9.5080    10.1426    51.5031 |    12.0082  51.5031 |      6.608
 60  256  

# Screening the Candidate for Total Module Dimensions

## MLWE-based Screen

The modulus exponent $b_q := \lceil \lg q \rceil$ is an input, read from `LG_Q_TARGET` (see `LG_Q_SOURCE` in "Input Preparations").

In [54]:
### ---- hard-core modulus exponent (input) ---------------------------------

def b_q_target(key):
    r"""b_q = ceil(lg q) for a key whose first two entries are (n, kappa), read from LG_Q_TARGET.

    Privacy places no lower bound on q, so b_q is an input: LG_Q_SOURCE selects the LaTSEC sizes or a fixed value.
    """
    return LG_Q_TARGET[(key[0], key[1])]

def q_target(key):
    r"""Representative modulus 2^b_q for the screen and the rough estimates."""
    return 2^b_q_target(key)

for g in GEOMETRY:
    for kappa in KAPPAS:
        assert q_target((g['n'], kappa)) == 2^LG_Q_TARGET[(g['n'], kappa)]

We consider the root Hermite factor $\delta_0(\beta) := \left(\frac{\beta}{2\pi e}(\pi\beta)^{1/\beta}\right)^{\frac{1}{2(\beta-1)}}$ attainable by BKZ with block size $\beta$ under the geometric series assumption and its heuristic relation $\lg\delta_0 \approx \frac{\lg^2(q/\sigma)}{4N\lg q}$.

In [55]:
### ---- step 12b: root-Hermite factor -------------------------------------

def delta_0(beta):
    r"""delta_0(beta) = ((beta/(2 pi e)) (pi beta)^(1/beta))^(1/(2(beta-1)))."""
    assert beta > 1
    b = PREC(beta)
    return ((b/(2*PI*PREC(1).exp())) * (PI*b)^(1/b))^(1/(2*(b-1)))

def lg_delta_0(beta):
    r"""lg delta_0(beta), as used in the screening relation."""
    return lg(delta_0(beta))

# Sanity: delta_0 is decreasing in beta, and delta_0(beta_kappa) sits in the range expected for these security levels.
assert all(delta_0(b) > delta_0(b+50) for b in (439, 658, 877))
assert PREC("1.0035") < delta_0(439) < PREC("1.0040"), delta_0(439)

With $N = \ell' d$ and $\sigma_\mathrm{MQ}(\ell) = S_\mathrm{MQ}(\kappa,\ell)$, we define
\begin{align*}
    \ell'_\mathrm{req}(\ell) := \frac{\lg^2\left(2^{b_q}/\sigma_\mathrm{MQ}(\ell)\right)}{4d\, b_q \lg\delta_0(\beta_\mathsf{MLWE})},
\end{align*}
where $\beta_\mathsf{MLWE} := \lceil \kappa_\mathsf{MLWE}/0.292\rceil$ and $\kappa_\mathsf{MLWE} = \kappa + 1$. Since $p = \ell' = \ell/2$, we discard $\ell$ whenever $\lceil \ell'_\mathrm{req}(\ell)\rceil > \ell/2$. This is a heuristic pre-screen only; the lattice estimator decides which candidates are retained.

Unlike the $\mathsf{LaTSEC}$ search, there is a single $\mathsf{MLWE}$ instance: there is no $\sigma_\mathrm{AK}$, and $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathbf{s}}$ is implied by the instance with $\sigma_\mathrm{MQ}$. The screen is keyed by $(n,\kappa,\ell)$: it does not depend on $Q$, and it depends on $n$ only through $b_q$.

In [56]:
### ---- the simplified MLWE screen -----------------------------------------
# Keys (n, kappa, ell): sigma_MQ = S_MQ(kappa, ell) does not depend on Q (checked in the
# previous section), and n enters only through b_q.

SCREEN_KEYS = [(g['n'], kappa, ell) for g in GEOMETRY for kappa in KAPPAS for ell in LIST_ELL]

def sigma_MQ_at(key):
    r"""sigma_MQ at key = (n, kappa, ell); identical for every Q, so it is read at Q = 0."""
    n, kappa, ell = key
    return SIGMA_MLWE[(n, kappa, 0, ell)]['sigma_MQ']

def ell_prime_req(key):
    r"""Minimum secret rank ell' for MLWE_{p, ell', 2^b_q, d, sigma_MQ} to reach kappa_MLWE,
    under lg delta_0 ~ lg^2(q/sigma) / (4 N lg q) with N = ell' d."""
    n, kappa, ell = key
    b_q = b_q_target(key)
    return lg(2^b_q / sigma_MQ_at(key))^2 / (4*d*b_q*lg_delta_0(BETA_MLWE[kappa]))

ELL_REQ       = {key: ell_prime_req(key) for key in SCREEN_KEYS}
ELL_PRIME_MIN = {key: max(1, ceil(ELL_REQ[key])) for key in SCREEN_KEYS}
PASS_MLWE     = {key: ELL_PRIME_MIN[key] <= key[2] // 2 for key in SCREEN_KEYS}   # ell' = ell/2

Consistency check.

In [57]:
for key in SCREEN_KEYS:
    assert ELL_REQ[key] > 0, f"{key}: ell_req = {ELL_REQ[key]}"
    assert PASS_MLWE[key] == (ELL_PRIME_MIN[key] <= key[2] // 2), key

# Monotone in ell: sigma_MQ grows with ell, so ell'_req does not increase, while ell/2 does.
for g in GEOMETRY:
    for kappa in KAPPAS:
        passed = [PASS_MLWE[(g['n'], kappa, ell)] for ell in sorted(LIST_ELL)]
        assert passed == sorted(passed), f"(n, kappa) = ({g['n']}, {kappa}): screen not monotone in ell"

First passing $\ell$ and table generator.

In [58]:
### ---- screen results ----------------------------------------------------

def screen_MLWE_table(L=None, only_pass=False):
    r"""Print the simplified MLWE screen for ell in L (default: all of LIST_ELL)."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | {'b_q':>4} {'lg sig_MQ':>10} {'ell_req':>9} | "
          f"{'ell_min':>8} {'ell_p':>6} {'pass':>5}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                key = (g['n'], kappa, ell)
                if only_pass and not PASS_MLWE[key]:
                    continue
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | {b_q_target(key):>4} "
                      f"{float(lg(sigma_MQ_at(key))):>10.4f} {float(ELL_REQ[key]):>9.4f} | "
                      f"{ELL_PRIME_MIN[key]:>8} {ell//2:>6} {('yes' if PASS_MLWE[key] else 'no'):>5}")

def first_passing_ell():
    r"""Smallest ell in LIST_ELL passing the MLWE screen, per (n, kappa)."""
    print(f"{'n':>3} {'kappa':>6} {'b_q':>4} {'first ell':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            hit = next((ell for ell in sorted(LIST_ELL) if PASS_MLWE[(g['n'], kappa, ell)]), None)
            print(f"{g['n']:>3} {kappa:>6} {LG_Q_TARGET[(g['n'], kappa)]:>4} {str(hit):>10}")

Printing a table.

In [59]:
first_passing_ell()
print()
screen_MLWE_table([4, 6, 8, 10])

  n  kappa  b_q  first ell
 40    128   32          6
 40    192   32          8
 40    256   32         10
 60    128   32          6
 60    192   32          8
 60    256   32         10

  n  kap  ell |  b_q  lg sig_MQ   ell_req |  ell_min  ell_p  pass
 40  128    4 |   32     1.6641    2.6255 |        3      2    no
 40  128    6 |   32     1.6670    2.6250 |        3      3   yes
 40  128    8 |   32     1.6691    2.6246 |        3      4   yes
 40  128   10 |   32     1.6707    2.6243 |        3      5   yes
 40  192    4 |   32     1.9309    3.4419 |        4      2    no
 40  192    6 |   32     1.9329    3.4414 |        4      3    no
 40  192    8 |   32     1.9343    3.4411 |        4      4   yes
 40  192   10 |   32     1.9355    3.4409 |        4      5   yes
 40  256    4 |   32     2.1252    4.2037 |        5      2    no
 40  256    6 |   32     2.1268    4.2033 |        5      3    no
 40  256    8 |   32     2.1279    4.2030 |        5      4    no
 40  256   10 |   

# Enumerate the Admissible Module Splits

With $p = \ell' = \ell/2$ fixed, each even $\ell$ gives exactly one split, so the enumeration reduces to a list of candidate values of $\ell$ in ascending order. The list contains every $\ell$ that passes the $\mathsf{MLWE}$ screen, together with the largest $\ell$ below the first passing one: the screen is heuristic, so the boundary is decided by the lattice estimator in the next section. Unlike the $\mathsf{LaTSEC}$ search, there is no $\mathsf{MSIS}$ priority flag and no upper cap on $p$.

In [60]:
### ---- enumerate the candidate module dimensions --------------------------
# One split per even ell: p = ell' = ell/2.

def split_list(n, kappa, extra_below=1):
    r"""Candidates (ell, p, ell', screened) for one (n, kappa), ascending in ell.

    Every ell passing the MLWE screen, plus the `extra_below` largest values of ell just
    below the first passing one, so that the estimator decides the boundary.
    The share size grows with ell, so the first candidate retained by the estimator is
    the size-optimal one.
    """
    ells = sorted(LIST_ELL)
    passing = [ell for ell in ells if PASS_MLWE[(n, kappa, ell)]]
    if not passing:
        return []
    below = [ell for ell in ells if ell < passing[0]]
    below = below[-extra_below:] if extra_below > 0 else []
    return [(ell, ell // 2, ell // 2, PASS_MLWE[(n, kappa, ell)]) for ell in below + passing]

SPLIT_LIST = {(g['n'], kappa): split_list(g['n'], kappa) for g in GEOMETRY for kappa in KAPPAS}

for (n, kappa), lst in SPLIT_LIST.items():
    assert lst, f"(n, kappa) = ({n}, {kappa}): no candidate passes the screen; extend LIST_ELL"
    for ell, p, lp, scr in lst:
        assert p + lp == ell and p == lp, (n, kappa, ell)
        assert scr == PASS_MLWE[(n, kappa, ell)], (n, kappa, ell)
    ells = [t[0] for t in lst]
    assert ells == sorted(set(ells)), (n, kappa)
    assert sum(1 for t in lst if not t[3]) <= 1, (n, kappa)     # only the extra one below

Function for printing.

In [61]:
def split_list_table(n, kappa):
    r"""Print the candidate list for one (n, kappa)."""
    lst = SPLIT_LIST[(n, kappa)]
    print(f"n={n} kappa={kappa}: {len(lst)} candidates")
    print(f"  {'#':>3} {'ell':>4} {'p':>4} {'ell_p':>6} {'screen':>7}")
    for i, (ell, p, lp, scr) in enumerate(lst):
        print(f"  {i:>3} {ell:>4} {p:>4} {lp:>6} {('pass' if scr else 'below'):>7}")

def split_list_summary():
    r"""Candidate counts, the first candidate, and the first screened ell, per (n, kappa)."""
    print(f"{'n':>3} {'kappa':>6} {'cands':>6} {'screened':>9} {'first cand':>11} {'first pass':>11}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            lst = SPLIT_LIST[(g['n'], kappa)]
            nscr = sum(1 for t in lst if t[3])
            first_c = lst[0][0] if lst else None
            first_p = next((t[0] for t in lst if t[3]), None)
            print(f"{g['n']:>3} {kappa:>6} {len(lst):>6} {nscr:>9} {str(first_c):>11} {str(first_p):>11}")

Summary results.

In [62]:
split_list_summary()
print()
split_list_table(40, 256)

  n  kappa  cands  screened  first cand  first pass
 40    128     11        10           4           6
 40    192     10         9           6           8
 40    256      9         8           8          10
 60    128     11        10           4           6
 60    192     10         9           6           8
 60    256      9         8           8          10

n=40 kappa=256: 9 candidates
    #  ell    p  ell_p  screen
    0    8    4      4   below
    1   10    5      5    pass
    2   12    6      6    pass
    3   14    7      7    pass
    4   16    8      8    pass
    5   18    9      9    pass
    6   20   10     10    pass
    7   22   11     11    pass
    8   24   12     12    pass


# Estimate the MLWE Problem at the Fixed Modulus

## Iteration Preparation

We select the largest prime $q < 2^{b_q}$, with $q \equiv 1 \pmod{2d}$ when NTT compatibility is required, so that $\lceil\lg q\rceil = b_q$. Unlike the $\mathsf{LaTSEC}$ search, there is no lower bound $q > 2B_\mathsf{MSIS}+1$, since there is no $\mathsf{MSIS}$ instance; for the $\mathsf{LaTSEC}$ moduli, the rule reproduces the $\mathsf{LaTSEC}$ primes. The rough estimates are computed at the representative modulus $2^{b_q}$. For a retained candidate, the full estimate is recomputed at the exact prime. This prime lies within relative distance $2^{-18}$ of $2^{b_q}$ for every $b_q$ used here (for $b_q = 32$, $q = 2^{32} - 10239$), so $\lg q$ and $b_q$ differ by less than $10^{-5}$.

Prime selection for later use.

In [63]:
### ---- prime selection ----------------------------------------------------

def first_admissible_prime(b_q):
    r"""Largest prime q < 2^b_q, with q = 1 mod 2d if NTT_REQUIRED, so that ceil(lg q) = b_q.

    There is no lower bound from MSIS; for the LaTSEC moduli this rule reproduces the LaTSEC primes.
    """
    m = 2*d if NTT_REQUIRED else 1
    q = 2^b_q - 1                                 # largest candidate below 2^b_q
    q -= (q - 1) % m                              # largest q < 2^b_q with q = 1 mod m
    while not is_prime(q):
        q -= m
        assert q > 2^(b_q - 1), f"no admissible prime between 2**{b_q - 1} and 2**{b_q}"
    assert (not NTT_REQUIRED) or q % (2*d) == 1, "q != 1 mod 2d"
    assert Integer(q).nbits() == b_q, "ceil(lg q) != b_q"
    return q

Some testing.

In [64]:
PRIMES = {b: first_admissible_prime(b) for b in sorted(set(LG_Q_TARGET.values()))}
for b, q in PRIMES.items():
    print(f"b_q = {b}:  q = {q}")

b_q = 32:  q = 4294957057


We downcast $\sigma_\mathrm{MQ}$ and $\sigma_\mathbf{s}$ from PREC to float, since `ND.DiscreteGaussian` is NumPy-backed; the modulus $q$ stays an exact integer. The empty-view instance uses the smallest $\sigma_\mathbf{s}$ over the query budgets: since the estimate does not decrease with the width, this one estimate covers every $Q$.

In [65]:
### ---- cross the precision boundary ---------------------------------------
# The estimator is NumPy-backed, so real-valued inputs are float64 there.
# This is the one deliberate downcast: beta moves by ~6.4 per bit of lg(sigma), so a 2^-53
# relative error cannot shift it.  q is NOT downcast: a prime such as 2^61 - 10239 needs
# 61 mantissa bits and would round back to 2^61.

def sigma_s_min(key):
    r"""Smallest sigma_s over Q at key = (n, kappa, ell).

    The empty-view instance MLWE_{p, ell', q, d, sigma_s} is estimated only at this width;
    the estimate does not decrease with the width, so it covers every Q.
    """
    n, kappa, ell = key
    return min(WIDTHS[(n, kappa, Q, ell)]['sigma_s'] for Q in QUERIES)

def estimator_inputs(key):
    r"""(sigma_MQ, sigma_s_min) at key = (n, kappa, ell), downcast to float for the estimator."""
    return float(sigma_MQ_at(key)), float(sigma_s_min(key))

We estimate $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}(\ell)}$ for nonempty auxiliary views and $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathbf{s}}$ for the empty view, with $p = \ell' = \ell/2$, using `MLWE_to_LWE_params(n_rows, m_cols, q, d, sigma)` with input $(p,\ell',q,d,\sigma)$.

In [66]:
### ---- build the two estimator parameter objects --------------------------

def build_params(key, q=None):
    r"""The two MLWE instances at key = (n, kappa, ell), ready for the estimator.

        MQ: MLWE_{p, ell', q, d, sigma_MQ}      -> LWE_{ell' d, p d, q, sigma_MQ}      (nonempty views)
        S : MLWE_{p, ell', q, d, sigma_s_min}   -> LWE_{ell' d, p d, q, sigma_s_min}   (empty view)

    with p = ell' = ell/2.  q defaults to the representative 2^b_q; pass an exact prime
    to re-estimate a retained candidate.  q stays an exact Integer.
    """
    n, kappa, ell = key
    p = lp = ell // 2
    q = q_target(key) if q is None else q
    assert q in ZZ and q >= 2, "q must stay an exact integer"
    s_MQ, s_S = estimator_inputs(key)
    stem = f"n={n},kap={kappa},ell={ell},p={p},lp={lp},bq={Integer(q - 1).nbits()}"
    return dict(
        MQ = MLWE_to_LWE_params(p, lp, q, d, s_MQ, tag=f"MLWE-MQ {stem}"),
        S  = MLWE_to_LWE_params(p, lp, q, d, s_S,  tag=f"MLWE-S {stem}"),
    )

Some testing.

In [67]:
# Test at the first screened ell for (n, kappa) = (40, 128); reused by the spot checks below.
ELL_TEST = next(ell for ell, p, lp, scr in SPLIT_LIST[(40, 128)] if scr)
P = build_params((40, 128, ELL_TEST))
print(P['MQ'])    # n = ell'*d (secret dim), m = p*d (samples)
print(P['S'])

LWEParameters(n=1536, q=4294967296, Xs=D(σ=3.18), Xe=D(σ=3.18), m=1536, tag='MLWE-MQ n=40,kap=128,ell=6,p=3,lp=3,bq=32')
LWEParameters(n=1536, q=4294967296, Xs=D(σ=7.74), Xe=D(σ=7.74), m=1536, tag='MLWE-S n=40,kap=128,ell=6,p=3,lp=3,bq=32')


We define $\mathsf{Sec}$, the concrete bit-security of one $\mathsf{MLWE}$ instance (lg of the cheapest finite `rop`). The same instance is queried by the sweep, the validation, and the headroom bisection, so `Sec_at` caches each estimate by (key, instance, modulus, rough).

In [68]:
### ---- Sec of one instance, with caching ----------------------------------

def Sec(params, rough=False, cost_model=RC.ADPS16, verbose=False):
    r"""Concrete bit-security of one MLWE instance: lg of the cheapest finite rop, or None.

    rough=True uses LWE.estimate.rough (usvp and dual_hybrid only, ADPS16 hard-coded), whose
    value upper-bounds the full estimate: valid for pruning, never for retention.
    """
    if rough:
        _, _, _, s = run_LWE_rough_analysis(params, verbose=verbose)
    else:
        _, _, _, s = run_LWE_full_analysis(params, cost_model=cost_model, verbose=verbose)
    return s

@cached_function
def Sec_at(key, which, q, rough):
    r"""Cached Sec of instance `which` in {'MQ', 'S'} at key = (n, kappa, ell) and exact modulus q."""
    assert which in ('MQ', 'S')
    return Sec(build_params(key, q=q)[which], rough=rough)

Retention test. A candidate is retained when $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}}$ reaches $\kappa_\mathsf{MLWE} = \kappa + 1$ bits. The empty-view instance $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathbf{s}}$ is implied, since $\sigma_\mathrm{MQ} \leq \sigma_\mathbf{s}/\sqrt{2}$ at every grid point and the estimated security does not decrease with the width. Unlike the $\mathsf{LaTSEC}$ search, there is no $\mathsf{MSIS}$ target and no separate abort-key target. A `None` value of $\mathsf{Sec}$ means that no attack was costed, which indicates a degenerate instance rather than a hard one, so it is treated as a failure.

In [69]:
def target(key):
    r"""kappa_MLWE = kappa + 1 for a key whose second entry is kappa."""
    return KAPPA_MLWE[key[1]]

def is_retained(key, secs):
    r"""True iff every value in secs (a dict over instances) reaches kappa_MLWE.
    A None Sec means no attack was costed; it is treated as a failure."""
    t = target(key)
    return all(s is not None and s >= t for s in secs.values())

Some testing.

In [70]:
%%time
key = (40, 128, ELL_TEST)
s_rough = Sec_at(key, 'MQ', q_target(key), True)
s_full  = Sec_at(key, 'MQ', q_target(key), False)
print(f"MQ at {key}, q = 2**{b_q_target(key)}: rough = {s_rough:.1f}   full = {s_full:.1f}   "
      f"target = {target(key)}   {'ok' if s_full >= target(key) else 'FAIL'}")

usvp                 :: rop: ≈2^150.4, red: ≈2^150.4, δ: 1.003333, β: 515, d: 3047, tag: usvp
dual_hybrid          :: rop: ≈2^152.4, red: ≈2^152.4, guess: ≈2^121.7, β: 522, p: 3, ζ: 0, t: 0, β': 522, N: ≈2^108.0, m: 1536
usvp                 :: rop: ≈2^150.4, red: ≈2^150.4, δ: 1.003333, β: 515, d: 3047, tag: usvp
bdd                  :: rop: ≈2^151.1, red: ≈2^150.4, svp: ≈2^149.8, β: 515, η: 512, d: 3065, tag: bdd
dual                 :: rop: ≈2^153.0, mem: ≈2^117.8, m: 1536, β: 524, d: 3072, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^152.4, red: ≈2^152.4, guess: ≈2^121.7, β: 522, p: 3, ζ: 0, t: 0, β': 522, N: ≈2^108.0, m: 1536
MQ at (40, 128, 6), q = 2**32: rough = 150.4   full = 150.4   target = 129   ok
CPU times: user 30 s, sys: 288 ms, total: 30.3 s
Wall time: 30.3 s


Some checking (the attacks from LWE analysis).

In [71]:
%%time
res, _, _, _ = run_LWE_full_analysis(P['MQ'], verbose=False)
print(sorted(res.keys()))

usvp                 :: rop: ≈2^150.4, red: ≈2^150.4, δ: 1.003333, β: 515, d: 3047, tag: usvp
bdd                  :: rop: ≈2^151.1, red: ≈2^150.4, svp: ≈2^149.8, β: 515, η: 512, d: 3065, tag: bdd
dual                 :: rop: ≈2^153.0, mem: ≈2^117.8, m: 1536, β: 524, d: 3072, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^152.4, red: ≈2^152.4, guess: ≈2^121.7, β: 522, p: 3, ζ: 0, t: 0, β': 522, N: ≈2^108.0, m: 1536
['bdd', 'bdd_hybrid', 'bdd_mitm_hybrid', 'dual', 'dual_hybrid', 'usvp']
CPU times: user 3.89 s, sys: 10.1 ms, total: 3.9 s
Wall time: 3.9 s


## The Iteration

For each $(n,\kappa)$, we walk the candidate list in ascending order of $\ell$ and run the rough estimate of $\mathsf{MLWE}_{\ell/2,\ell/2,2^{b_q},d,\sigma_\mathrm{MQ}(\ell)}$. Since the share size grows with $\ell$, the first $\ell$ whose rough estimate reaches $\kappa_\mathsf{MLWE}$ is the size-optimal candidate.

Unlike the $\mathsf{LaTSEC}$ search, the modulus is fixed, so there is no inner sweep over $b_q$ and no $\mathsf{MSIS}$ check. Only the instance with $\sigma_\mathrm{MQ}$ is estimated: since $\sigma_\mathbf{s} \geq \sqrt{2}\sigma_\mathrm{MQ}$, the requirement for the empty-view instance is implied. The rough estimate upper-bounds the full estimate, so a candidate passing here may still fail validation; in that case, the validation step moves to the next candidate.

The first candidate in each list is the largest $\ell$ below the first screened one. We assert that it fails the rough check, unless it is the smallest $\ell$ in the grid, so that the estimator, not the heuristic screen, decides the boundary.

In [72]:
### ---- rough check of one candidate ---------------------------------------

def rough_check(key):
    r"""Rough Sec of MLWE_{ell/2, ell/2, 2^b_q, d, sigma_MQ} at key = (n, kappa, ell).

    Returns dict(Sec_MQ_rough, pass_rough).  A None Sec counts as a failure.
    """
    s = Sec_at(key, 'MQ', q_target(key), True)
    return dict(Sec_MQ_rough=s, pass_rough=(s is not None and s >= target(key)))

Function for the sweep using rough analysis.

In [73]:
def sweep_rough(n, kappa, stop_at_first=True):
    r"""Walk SPLIT_LIST[(n, kappa)] in ascending ell with the rough check.

    stop_at_first=True stops at the first passing ell, which is the size-optimal candidate;
    the validation step falls back to later candidates in SPLIT_LIST if needed.
    Returns a list of dicts with ell, p, ell_prime, screened, Sec_MQ_rough, pass_rough.
    """
    out = []
    for ell, p, lp, scr in SPLIT_LIST[(n, kappa)]:
        r = rough_check((n, kappa, ell))
        out.append(dict(ell=ell, p=p, ell_prime=lp, screened=scr, **r))
        if stop_at_first and r['pass_rough']:
            break
    return out

Printing the candidates.

In [74]:
%%time
ROUGH = {(g['n'], kappa): sweep_rough(g['n'], kappa) for g in GEOMETRY for kappa in KAPPAS}

for (n, kappa), hits in ROUGH.items():
    assert hits and hits[-1]['pass_rough'], \
        f"(n, kappa) = ({n}, {kappa}): no candidate passes the rough check; extend LIST_ELL"
    # The estimator decides the boundary only if the first candidate fails (or is the smallest ell).
    assert not hits[0]['pass_rough'] or hits[0]['ell'] == min(LIST_ELL), \
        f"(n, kappa) = ({n}, {kappa}): the first candidate ell = {hits[0]['ell']} already passes; " \
        f"rebuild SPLIT_LIST with a larger extra_below"

usvp                 :: rop: ≈2^86.7, red: ≈2^86.7, δ: 1.004875, β: 297, d: 2041, tag: usvp
dual_hybrid          :: rop: ≈2^88.2, red: ≈2^88.2, guess: ≈2^74.9, β: 302, p: 3, ζ: 0, t: 0, β': 302, N: ≈2^61.2, m: 1024
usvp                 :: rop: ≈2^153.9, red: ≈2^153.9, δ: 1.003279, β: 527, d: 3068, tag: usvp
dual_hybrid          :: rop: ≈2^156.2, red: ≈2^156.2, guess: ≈2^124.0, β: 535, p: 3, ζ: 0, t: 0, β': 535, N: ≈2^109.8, m: 1536
usvp                 :: rop: ≈2^223.1, red: ≈2^223.1, δ: 1.002500, β: 764, d: 4056, tag: usvp
dual_hybrid          :: rop: ≈2^225.1, red: ≈2^225.1, guess: ≈2^215.4, β: 771, p: 3, ζ: 10, t: 0, β': 771, N: ≈2^158.6, m: ≈2^11.0
usvp                 :: rop: ≈2^226.6, red: ≈2^226.6, δ: 1.002472, β: 776, d: 4093, tag: usvp
dual_hybrid          :: rop: ≈2^228.9, red: ≈2^228.9, guess: ≈2^220.6, β: 784, p: 3, ζ: 10, t: 0, β': 784, N: ≈2^161.5, m: ≈2^11.0
usvp                 :: rop: ≈2^299.6, red: ≈2^299.6, δ: 1.002004, β: 1026, d: 5115, tag: usvp
dual_hybrid        

Printing the candidate from ROUGH.

In [75]:
for (n, kappa), hits in ROUGH.items():
    print(f"n={n} kappa={kappa} b_q={LG_Q_TARGET[(n, kappa)]} target={KAPPA_MLWE[kappa]}:")
    for h in hits:
        s = "None" if h['Sec_MQ_rough'] is None else "%.1f" % h['Sec_MQ_rough']
        print(f"   ell={h['ell']:>2} p={h['p']:>2} l'={h['ell_prime']:>2} "
              f"screen={'pass' if h['screened'] else 'below':<5}  rough Sec_MQ={s:>6}  "
              f"{'PASS' if h['pass_rough'] else 'fail'}")

n=40 kappa=128 b_q=32 target=129:
   ell= 4 p= 2 l'= 2 screen=below  rough Sec_MQ=  86.7  fail
   ell= 6 p= 3 l'= 3 screen=pass   rough Sec_MQ= 150.4  PASS
n=40 kappa=192 b_q=32 target=193:
   ell= 6 p= 3 l'= 3 screen=below  rough Sec_MQ= 153.9  fail
   ell= 8 p= 4 l'= 4 screen=pass   rough Sec_MQ= 223.1  PASS
n=40 kappa=256 b_q=32 target=257:
   ell= 8 p= 4 l'= 4 screen=below  rough Sec_MQ= 226.6  fail
   ell=10 p= 5 l'= 5 screen=pass   rough Sec_MQ= 299.6  PASS
n=60 kappa=128 b_q=32 target=129:
   ell= 4 p= 2 l'= 2 screen=below  rough Sec_MQ=  86.7  fail
   ell= 6 p= 3 l'= 3 screen=pass   rough Sec_MQ= 150.4  PASS
n=60 kappa=192 b_q=32 target=193:
   ell= 6 p= 3 l'= 3 screen=below  rough Sec_MQ= 153.9  fail
   ell= 8 p= 4 l'= 4 screen=pass   rough Sec_MQ= 223.1  PASS
n=60 kappa=256 b_q=32 target=257:
   ell= 8 p= 4 l'= 4 screen=below  rough Sec_MQ= 226.6  fail
   ell=10 p= 5 l'= 5 screen=pass   rough Sec_MQ= 299.6  PASS


# Validate and Rank the Surviving Candidates

Validate the rough analysis result. For each candidate, we re-estimate $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}}$ with the full estimator (cost model ADPS16, excluding BKW and Arora-GB) at the exact prime $q < 2^{b_q}$, and retain the candidate if the estimate reaches $\kappa_\mathsf{MLWE} = \kappa + 1$. The empty-view instance $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathbf{s}}$ is not estimated separately: $\sigma_\mathrm{MQ} \leq \sigma_\mathbf{s}/\sqrt{2}$ at every grid point (asserted when $\sigma_\mathrm{MQ}$ was computed), and the estimated security does not decrease with the width, so its requirement is implied.

For the retained candidate, we also compute the headroom $b_q^{\max}$, the largest $b$ with $\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',2^b,d,\sigma_\mathrm{MQ}}) \geq \kappa_\mathsf{MLWE}$. Since the estimate does not increase with $q$, the passing exponents form an interval $[b_q, b_q^{\max}]$: we bisect with the rough estimate and then confirm with the full estimate, stepping down while it fails. Estimating at $q = 2^b$ covers every $q \leq 2^b$, so the parameter set is private for every hard-core modulus $q < 2^{b_q^{\max}}$.

In [76]:
### ---- validate one candidate, and its modulus headroom --------------------

def validate_hardness(key):
    r"""Full estimate of MLWE_{ell/2, ell/2, q, d, sigma_MQ} at the exact prime, for key = (n, kappa, ell).

    The empty-view instance with sigma_s is implied, since sigma_MQ <= sigma_s/sqrt(2) and the
    estimate does not decrease with the width.  Returns a dict with the split, b_q, the prime q,
    Sec_MQ, its margin over kappa_MLWE, and a retained flag.
    """
    n, kappa, ell = key
    b_q = b_q_target(key)
    q = PRIMES[b_q]
    s = Sec_at(key, 'MQ', q, False)
    t = target(key)
    return dict(ell=ell, p=ell//2, ell_prime=ell//2, b_q=b_q, q=q,
                Sec_MQ=s, margin=(None if s is None else s - t),
                retained=is_retained(key, {'MQ': s}))

def headroom_b_q(key, b_hi=None):
    r"""Largest b in [b_q, b_hi] with Sec(MLWE_{ell/2, ell/2, 2^b, d, sigma_MQ}) >= kappa_MLWE.

    The estimate does not increase with q, so the passing exponents form an interval [b_q, b_max].
    Bisection with the rough estimate locates b_max; the full estimate then confirms it, stepping
    down while it fails, since the rough estimate upper-bounds the full one.
    Returns (b_max, full Sec at q = 2^b_max).
    """
    b_hi = BQ_HEADROOM_MAX if b_hi is None else b_hi
    t = target(key)

    def ok(b, rough):
        s = Sec_at(key, 'MQ', 2^b, rough)
        return s is not None and s >= t

    lo, hi = b_q_target(key), b_hi
    assert ok(lo, True), f"{key}: rough estimate fails at b_q itself"
    if ok(hi, True):
        lo = hi
    while hi - lo > 1:                        # invariant: ok(lo), not ok(hi)
        mid = (lo + hi) // 2
        if ok(mid, True):
            lo = mid
        else:
            hi = mid
    b = lo
    while not ok(b, False):
        b -= 1
        assert b >= b_q_target(key), f"{key}: full estimate fails at b_q"
    return b, Sec_at(key, 'MQ', 2^b, False)

Validate the grid, one $(n,\kappa)$ at a time.

In [77]:
### ---- validate the grid, one (n, kappa) at a time ------------------------
# FINAL accumulates across cells.  Run this cell once; re-running it discards
# results already computed (the estimates themselves stay cached in Sec_at).

FINAL = {}

def show_row(n, kappa, r):
    r"""One line per validated candidate, with the margin over kappa_MLWE."""
    s = "None" if r['Sec_MQ'] is None else "%.1f (%+.1f)" % (r['Sec_MQ'], r['margin'])
    line = (f"n={n} kap={kappa}: ell={r['ell']} p={r['p']} l'={r['ell_prime']} b_q={r['b_q']} | "
            f"MQ={s}  {'RETAINED' if r['retained'] else 'FAILED'}")
    if r['retained']:
        line += f"  | b_q_max={r['b_q_max']} (Sec_MQ={r['Sec_MQ_at_b_q_max']:.1f})"
    print(line)

def validate_key(n, kappa):
    r"""Validate the candidates for one (n, kappa), starting at the first rough pass.

    Walks SPLIT_LIST from the first ell that passed the rough check; the first ell whose full estimate reaches kappa_MLWE is retained, 
    and its headroom b_q_max is computed.
    Writes FINAL[(n, kappa)] (all validated rows, the last one retained) and prints one line per row.
    """
    ell0 = next(h['ell'] for h in ROUGH[(n, kappa)] if h['pass_rough'])
    rows = []
    for ell, p, lp, scr in SPLIT_LIST[(n, kappa)]:
        if ell < ell0:
            continue
        r = validate_hardness((n, kappa, ell))
        if r['retained']:
            r['b_q_max'], r['Sec_MQ_at_b_q_max'] = headroom_b_q((n, kappa, ell))
        rows.append(r)
        show_row(n, kappa, r)
        if r['retained']:
            break
    assert rows and rows[-1]['retained'], \
        f"(n, kappa) = ({n}, {kappa}): no candidate retained; extend LIST_ELL"
    FINAL[(n, kappa)] = rows
    return rows

The final table.

In [78]:
def final_table():
    r"""All validated candidates, in profile/kappa order."""
    print(f"{'n':>3} {'kap':>4} {'ell':>4} {'p':>3} {'l_p':>4} {'b_q':>4} | "
          f"{'Sec_MQ':>8} | {'b_q_max':>8} | {'ok':>4}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                bmax = str(r.get('b_q_max', '--'))
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} {r['p']:>3} {r['ell_prime']:>4} {r['b_q']:>4} | "
                      f"{r['Sec_MQ']:>8.1f} | {bmax:>8} | "
                      f"{('yes' if r['retained'] else 'NO'):>4}")
    missing = [(g['n'], kappa) for g in GEOMETRY for kappa in KAPPAS if (g['n'], kappa) not in FINAL]
    if missing:
        print(f"\nnot yet validated: {missing}")

Some testing at $(n,\kappa) = (40,128)$ and the first screened $\ell$ (`ELL_TEST`).

In [79]:
%%time
# Spot check of the width-monotonicity argument at one grid point (not used for retention).
key = (40, 128, ELL_TEST)
q = PRIMES[b_q_target(key)]
print(f"sigma_MQ = {float(sigma_MQ_at(key)):.3f},  sigma_s_min = {float(sigma_s_min(key)):.3f}")
print(f"Sec_MQ = {Sec_at(key, 'MQ', q, False):.1f},  Sec_S = {Sec_at(key, 'S', q, False):.1f}")

sigma_MQ = 3.176,  sigma_s_min = 7.739
usvp                 :: rop: ≈2^150.4, red: ≈2^150.4, δ: 1.003333, β: 515, d: 3047, tag: usvp
bdd                  :: rop: ≈2^151.1, red: ≈2^150.4, svp: ≈2^149.8, β: 515, η: 512, d: 3065, tag: bdd
dual                 :: rop: ≈2^153.0, mem: ≈2^117.8, m: 1536, β: 524, d: 3072, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^152.4, red: ≈2^152.4, guess: ≈2^121.7, β: 522, p: 3, ζ: 0, t: 0, β': 522, N: ≈2^108.0, m: 1536
usvp                 :: rop: ≈2^169.4, red: ≈2^169.4, δ: 1.003060, β: 580, d: 3053, tag: usvp
bdd                  :: rop: ≈2^169.9, red: ≈2^169.4, svp: ≈2^168.2, β: 580, η: 575, d: 3072, tag: bdd
dual                 :: rop: ≈2^172.6, mem: ≈2^131.8, m: 1536, β: 591, d: 3072, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^172.0, red: ≈2^172.0, guess: ≈2^135.7, β: 589, p: 3, ζ: 0, t: 0, β': 589, N: ≈2^119.4, m: 1536
Sec_MQ = 150.4,  Sec_S = 169.4
CPU times: user 52.5 s, sys: 308 ms, total: 52.8 s
Wall time: 52.8 s


## Validating at $n = 40$ and $\kappa = 128$.

In [80]:
%%time
validate_key(40, 128)

usvp                 :: rop: ≈2^11.7, red: ≈2^11.7, δ: 1.012950, β: 40, d: 2498, tag: usvp
dual_hybrid          :: rop: ≈2^13.7, red: ≈2^12.0, guess: ≈2^13.2, β: 41, p: 3, ζ: 0, t: 0, β': 41, N: 0.695, m: 1536
usvp                 :: rop: ≈2^34.2, red: ≈2^34.2, δ: 1.008548, β: 117, d: 3061, tag: usvp
dual_hybrid          :: rop: ≈2^35.0, red: ≈2^34.7, guess: ≈2^32.4, β: 119, p: 3, ζ: 0, t: 0, β': 119, N: ≈2^18.7, m: 1536
usvp                 :: rop: ≈2^64.2, red: ≈2^64.2, δ: 1.005920, β: 220, d: 3070, tag: usvp
dual_hybrid          :: rop: ≈2^65.1, red: ≈2^65.1, guess: ≈2^55.4, β: 223, p: 3, ζ: 0, t: 0, β': 223, N: ≈2^41.7, m: 1536
usvp                 :: rop: ≈2^94.0, red: ≈2^94.0, δ: 1.004618, β: 322, d: 3060, tag: usvp
dual_hybrid          :: rop: ≈2^95.2, red: ≈2^95.2, guess: ≈2^79.1, β: 326, p: 3, ζ: 0, t: 0, β': 326, N: ≈2^65.4, m: 1536
usvp                 :: rop: ≈2^117.4, red: ≈2^117.4, δ: 1.003968, β: 402, d: 3019, tag: usvp
dual_hybrid          :: rop: ≈2^118.8, red: ≈2^118.

[{'ell': 6,
  'p': 3,
  'ell_prime': 3,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 150.38,
  'margin': 21.379999999999995,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 132.27599999999998}]

## Validating at $n = 40$ and $\kappa = 192$.

In [81]:
%%time
validate_key(40, 192)

usvp                 :: rop: ≈2^223.1, red: ≈2^223.1, δ: 1.002500, β: 764, d: 4056, tag: usvp
bdd                  :: rop: ≈2^223.6, red: ≈2^223.1, svp: ≈2^221.9, β: 764, η: 759, d: 4095, tag: bdd
dual                 :: rop: ≈2^226.6, mem: ≈2^170.6, m: ≈2^11.0, β: 776, d: 4096, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^225.1, red: ≈2^225.1, guess: ≈2^215.4, β: 771, p: 3, ζ: 10, t: 0, β': 771, N: ≈2^158.6, m: ≈2^11.0
usvp                 :: rop: ≈2^22.8, red: ≈2^22.8, δ: 1.010374, β: 78, d: 3940, tag: usvp
dual_hybrid          :: rop: ≈2^23.5, red: ≈2^23.4, guess: ≈2^20.0, β: 80, p: 3, ζ: 0, t: 0, β': 80, N: 55.033, m: ≈2^11.0
usvp                 :: rop: ≈2^56.6, red: ≈2^56.6, δ: 1.006401, β: 194, d: 3977, tag: usvp
dual_hybrid          :: rop: ≈2^57.0, red: ≈2^56.9, guess: ≈2^51.1, β: 195, p: 3, ζ: 0, t: 0, β': 195, N: ≈2^36.9, m: ≈2^11.0
usvp                 :: rop: ≈2^99.6, red: ≈2^99.6, δ: 1.004443, β: 341, d: 4086, tag: usvp
dual_hybrid          :: rop: ≈2^100.4, red: ≈2^10

[{'ell': 8,
  'p': 4,
  'ell_prime': 4,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 223.088,
  'margin': 30.087999999999994,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 197.1}]

## Validating at $n = 40$ and $\kappa = 256$.

In [82]:
%%time
validate_key(40, 256)

usvp                 :: rop: ≈2^299.6, red: ≈2^299.6, δ: 1.002004, β: 1026, d: 5115, tag: usvp
bdd                  :: rop: ≈2^300.4, red: ≈2^299.9, svp: ≈2^298.7, β: 1027, η: 1022, d: 5111, tag: bdd
dual                 :: rop: ≈2^304.3, mem: ≈2^226.2, m: ≈2^11.3, β: 1042, d: 5120, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^302.8, red: ≈2^302.8, guess: ≈2^272.5, β: 1037, p: 3, ζ: 10, t: 0, β': 1037, N: ≈2^213.3, m: ≈2^11.3
usvp                 :: rop: ≈2^35.9, red: ≈2^35.9, δ: 1.008324, β: 123, d: 5028, tag: usvp
dual_hybrid          :: rop: ≈2^36.5, red: ≈2^36.5, guess: ≈2^28.3, β: 125, p: 3, ζ: 0, t: 0, β': 125, N: ≈2^13.7, m: ≈2^11.3
usvp                 :: rop: ≈2^80.3, red: ≈2^80.3, δ: 1.005129, β: 275, d: 5009, tag: usvp
dual_hybrid          :: rop: ≈2^80.6, red: ≈2^80.6, guess: ≈2^71.8, β: 276, p: 3, ζ: 0, t: 0, β': 276, N: ≈2^57.2, m: ≈2^11.3
usvp                 :: rop: ≈2^137.2, red: ≈2^137.2, δ: 1.003557, β: 470, d: 5104, tag: usvp
dual_hybrid          :: rop: ≈2^138.4

[{'ell': 10,
  'p': 5,
  'ell_prime': 5,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 299.592,
  'margin': 42.591999999999985,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 265.71999999999997}]

## Validating at $n = 60$ and $\kappa = 128$.

In [83]:
%%time
validate_key(60, 128)

usvp                 :: rop: ≈2^150.4, red: ≈2^150.4, δ: 1.003333, β: 515, d: 3047, tag: usvp
bdd                  :: rop: ≈2^151.1, red: ≈2^150.4, svp: ≈2^149.8, β: 515, η: 512, d: 3065, tag: bdd
dual                 :: rop: ≈2^153.0, mem: ≈2^117.8, m: 1536, β: 524, d: 3072, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^152.4, red: ≈2^152.4, guess: ≈2^121.7, β: 522, p: 3, ζ: 0, t: 0, β': 522, N: ≈2^108.0, m: 1536
usvp                 :: rop: ≈2^11.7, red: ≈2^11.7, δ: 1.012950, β: 40, d: 2498, tag: usvp
dual_hybrid          :: rop: ≈2^13.7, red: ≈2^12.0, guess: ≈2^13.2, β: 41, p: 3, ζ: 0, t: 0, β': 41, N: 0.695, m: 1536
usvp                 :: rop: ≈2^34.2, red: ≈2^34.2, δ: 1.008548, β: 117, d: 3061, tag: usvp
dual_hybrid          :: rop: ≈2^35.0, red: ≈2^34.7, guess: ≈2^32.4, β: 119, p: 3, ζ: 0, t: 0, β': 119, N: ≈2^18.7, m: 1536
usvp                 :: rop: ≈2^64.2, red: ≈2^64.2, δ: 1.005920, β: 220, d: 3070, tag: usvp
dual_hybrid          :: rop: ≈2^65.1, red: ≈2^65.1, guess: ≈2^5

[{'ell': 6,
  'p': 3,
  'ell_prime': 3,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 150.38,
  'margin': 21.379999999999995,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 132.27599999999998}]

## Validating at $n = 60$ and $\kappa = 192$.

In [84]:
%%time
validate_key(60, 192)

usvp                 :: rop: ≈2^223.1, red: ≈2^223.1, δ: 1.002500, β: 764, d: 4056, tag: usvp
bdd                  :: rop: ≈2^223.6, red: ≈2^223.1, svp: ≈2^221.9, β: 764, η: 759, d: 4095, tag: bdd
dual                 :: rop: ≈2^226.6, mem: ≈2^170.6, m: ≈2^11.0, β: 776, d: 4096, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^225.1, red: ≈2^225.1, guess: ≈2^215.4, β: 771, p: 3, ζ: 10, t: 0, β': 771, N: ≈2^158.6, m: ≈2^11.0
usvp                 :: rop: ≈2^22.8, red: ≈2^22.8, δ: 1.010374, β: 78, d: 3940, tag: usvp
dual_hybrid          :: rop: ≈2^23.5, red: ≈2^23.4, guess: ≈2^20.0, β: 80, p: 3, ζ: 0, t: 0, β': 80, N: 55.033, m: ≈2^11.0
usvp                 :: rop: ≈2^56.6, red: ≈2^56.6, δ: 1.006401, β: 194, d: 3977, tag: usvp
dual_hybrid          :: rop: ≈2^57.0, red: ≈2^56.9, guess: ≈2^51.1, β: 195, p: 3, ζ: 0, t: 0, β': 195, N: ≈2^36.9, m: ≈2^11.0
usvp                 :: rop: ≈2^99.6, red: ≈2^99.6, δ: 1.004443, β: 341, d: 4086, tag: usvp
dual_hybrid          :: rop: ≈2^100.4, red: ≈2^10

[{'ell': 8,
  'p': 4,
  'ell_prime': 4,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 223.088,
  'margin': 30.087999999999994,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 197.1}]

## Validating at $n = 60$ and $\kappa = 256$.

In [85]:
%%time
validate_key(60, 256)

usvp                 :: rop: ≈2^299.6, red: ≈2^299.6, δ: 1.002004, β: 1026, d: 5115, tag: usvp
bdd                  :: rop: ≈2^300.4, red: ≈2^299.9, svp: ≈2^298.7, β: 1027, η: 1022, d: 5111, tag: bdd
dual                 :: rop: ≈2^304.3, mem: ≈2^226.2, m: ≈2^11.3, β: 1042, d: 5120, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^302.8, red: ≈2^302.8, guess: ≈2^272.5, β: 1037, p: 3, ζ: 10, t: 0, β': 1037, N: ≈2^213.3, m: ≈2^11.3
usvp                 :: rop: ≈2^35.9, red: ≈2^35.9, δ: 1.008324, β: 123, d: 5028, tag: usvp
dual_hybrid          :: rop: ≈2^36.5, red: ≈2^36.5, guess: ≈2^28.3, β: 125, p: 3, ζ: 0, t: 0, β': 125, N: ≈2^13.7, m: ≈2^11.3
usvp                 :: rop: ≈2^80.3, red: ≈2^80.3, δ: 1.005129, β: 275, d: 5009, tag: usvp
dual_hybrid          :: rop: ≈2^80.6, red: ≈2^80.6, guess: ≈2^71.8, β: 276, p: 3, ζ: 0, t: 0, β': 276, N: ≈2^57.2, m: ≈2^11.3
usvp                 :: rop: ≈2^137.2, red: ≈2^137.2, δ: 1.003557, β: 470, d: 5104, tag: usvp
dual_hybrid          :: rop: ≈2^138.4

[{'ell': 10,
  'p': 5,
  'ell_prime': 5,
  'b_q': 32,
  'q': 4294957057,
  'Sec_MQ': 299.592,
  'margin': 42.591999999999985,
  'retained': True,
  'b_q_max': 35,
  'Sec_MQ_at_b_q_max': 265.71999999999997}]

## Validation per Query Budget

The estimator validation above runs once per $(n,\kappa)$, because the validated instance $\mathsf{MLWE}_{\ell/2,\ell/2,q,d,\sigma_\mathrm{MQ}}$ does not depend on $Q$: $\sigma_\mathrm{MQ} = S_\mathrm{MQ}(\kappa,\ell)$, while $q$ and the retained $\ell$ depend only on $(n,\kappa)$, and only on $\kappa$ when $b_q$ is fixed. The conditions that do depend on $Q$ are checked here for each of the twelve triples $(n,\kappa,Q)$ at the retained $\ell$, without further estimator calls:
1. posterior resampling, $R_\mathrm{MQ} = \sigma_\mathrm{MQ}/S_\mathrm{MQ} \geq 1$;
2. kernel smoothing, $R_\mathrm{ker} = \sigma_\mathrm{eff}/\tau_\mathrm{ker} \geq 1$;
3. the empty-view premise $\sigma_\mathrm{MQ} \leq \sigma_\mathbf{s}/\sqrt{2}$, reported as the gap $\lg\sigma_\mathbf{s} - \lg\sigma_\mathrm{MQ} \geq 1/2$.

In [86]:
### ---- validation per (n, kappa, Q) at the retained ell -------------------

def validation_by_Q():
    r"""One row per (n, kappa, Q) at the retained ell of FINAL[(n, kappa)].

    The MLWE estimate is shared by both values of Q; the smoothing ratios and the
    empty-view gap lg sigma_s - lg sigma_MQ (>= 1/2 required) are Q-specific.
    Returns the rows and asserts that every row passes.
    """
    half = PREC(1)/2
    rows = []
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} {'b_q':>4} | {'Sec_MQ':>7} {'target':>7} | "
          f"{'R_MQ':>7} {'R_ker':>7} {'gap_s':>7} | {'ok':>4}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            r = FINAL[(g['n'], kappa)][-1]                      # the retained row
            for Q in QUERIES:
                key = (g['n'], kappa, Q, r['ell'])
                s, w = SIGMA_MLWE[key], WIDTHS[key]
                gap = lg(w['sigma_s']) - lg(s['sigma_MQ'])
                ok = (r['retained'] and abs(s['R_MQ'] - 1) < TOL
                      and s['R_ker'] >= 1 - TOL and gap >= half)
                rows.append(dict(key=key, ok=ok))
                print(f"{g['n']:>3} {kappa:>4} {Q_label(Q):>5} {r['ell']:>4} {r['b_q']:>4} | "
                      f"{r['Sec_MQ']:>7.1f} {target(key):>7} | "
                      f"{float(s['R_MQ']):>7.4f} {float(s['R_ker']):>7.4f} {float(gap):>7.3f} | "
                      f"{('yes' if ok else 'NO'):>4}")
    assert all(x['ok'] for x in rows), "some (n, kappa, Q) fails validation"
    return rows

Validation by $Q$ output.

In [87]:
_ = validation_by_Q()

  n  kap     Q  ell  b_q |  Sec_MQ  target |    R_MQ   R_ker   gap_s |   ok
 40  128     0    6   32 |   150.4     129 |  1.0000  1.0000   1.285 |  yes
 40  128  2^64    6   32 |   150.4     129 |  1.0000  1.0000   2.057 |  yes
 40  192     0    8   32 |   223.1     193 |  1.0000  1.0000   1.298 |  yes
 40  192  2^64    8   32 |   223.1     193 |  1.0000  1.0000   2.101 |  yes
 40  256     0   10   32 |   299.6     257 |  1.0000  1.0000   1.305 |  yes
 40  256  2^64   10   32 |   299.6     257 |  1.0000  1.0000   2.125 |  yes
 60  128     0    6   32 |   150.4     129 |  1.0000  1.0000   0.791 |  yes
 60  128  2^64    6   32 |   150.4     129 |  1.0000  1.0000   0.998 |  yes
 60  192     0    8   32 |   223.1     193 |  1.0000  1.0000   0.795 |  yes
 60  192  2^64    8   32 |   223.1     193 |  1.0000  1.0000   1.006 |  yes
 60  256     0   10   32 |   299.6     257 |  1.0000  1.0000   0.797 |  yes
 60  256  2^64   10   32 |   299.6     257 |  1.0000  1.0000   1.011 |  yes


## The Retained Parameters

In [88]:
final_table()

  n  kap  ell   p  l_p  b_q |   Sec_MQ |  b_q_max |   ok
 40  128    6   3    3   32 |    150.4 |       35 |  yes
 40  192    8   4    4   32 |    223.1 |       35 |  yes
 40  256   10   5    5   32 |    299.6 |       35 |  yes
 60  128    6   3    3   32 |    150.4 |       35 |  yes
 60  192    8   4    4   32 |    223.1 |       35 |  yes
 60  256   10   5    5   32 |    299.6 |       35 |  yes


The modulus $q$ is not a parameter of $\mathsf{SEC}$ itself: $\mathsf{Setup}$, $\mathsf{Deal}$, and $\mathsf{Rec}$ work over $\mathbb{Z}$ and never reduce modulo $q$. It is the modulus of the hard-core function $f_{\bm{A}}(\bm{s}) = \bm{A}\bm{s} \bmod q$ in the privacy statement, so it belongs to the privacy instantiation and is reported in the parameter table as $\lceil\lg q\rceil = b_q$. No separate computation is needed, since `first_admissible_prime` asserts $\lceil\lg q\rceil = b_q$. We also report the headroom $b_q^{\max}$: the parameter set remains private for every hard-core modulus with $\lceil\lg q\rceil \leq b_q^{\max}$.

The modulus.

In [89]:
for (n, kappa), rows in FINAL.items():
    r = rows[-1]
    print(f"n={n} kap={kappa}: b_q={r['b_q']}  q={r['q']}  b_q_max={r['b_q_max']}  "
          f"headroom={r['b_q_max'] - r['b_q']} bits")

n=40 kap=128: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits
n=40 kap=192: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits
n=40 kap=256: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits
n=60 kap=128: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits
n=60 kap=192: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits
n=60 kap=256: b_q=32  q=4294957057  b_q_max=35  headroom=3 bits


Assembling the parameter.

In [90]:
### ---- assemble the parameter table ---------------------------------------

def parameter_row(n, kappa, Q, r):
    r"""One row of Table SEC-Selected-Parameters, from the retained FINAL entry r of (n, kappa).

    (ell, p, ell', q, b_q, b_q_max, Sec_MQ) are shared by both Q; the widths are Q-specific,
    and lg_sigma_r is None for Q = 0.
    """
    w = WIDTHS[(n, kappa, Q, r['ell'])]
    return dict(
        n=n, kappa=kappa, Q=Q, omega_c=OMEGA_C[kappa],
        lg_q=r['b_q'], q=r['q'], b_q_max=r['b_q_max'],
        ell=r['ell'], p=r['p'], ell_prime=r['ell_prime'],
        lg_sigma_s=lg(w['sigma_s']), lg_sigma_R=lg(w['sigma_R']),
        lg_sigma_r=(None if w['sigma_r'] is None else lg(w['sigma_r'])),
        Sec_MQ=r['Sec_MQ'],
    )

Printing parameter table.

In [91]:
PARAMS = [parameter_row(g['n'], kappa, Q, FINAL[(g['n'], kappa)][-1])
          for g in GEOMETRY for kappa in KAPPAS for Q in QUERIES]
assert len(PARAMS) == len(GEOMETRY) * len(KAPPAS) * len(QUERIES)
assert all(FINAL[(r['n'], r['kappa'])][-1]['retained'] for r in PARAMS)

def _num_or_dash(x, width, dec=4):
    r"""x with dec decimals, right-aligned in width, or '--' when x is None."""
    return ("--" if x is None else ("%." + str(dec) + "f") % float(x)).rjust(width)

def parameter_table():
    r"""Print the retained parameter sets, one row per (n, kappa, Q)."""
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'w_c':>4} {'lg q':>5} {'b_max':>6} {'ell':>4} {'p':>3} {'l_p':>4} | "
          f"{'lg sig_s':>9} {'lg sig_R':>9} {'lg sig_r':>9} | {'Sec_MQ':>7}")
    for r in PARAMS:
        print(f"{r['n']:>3} {r['kappa']:>4} {Q_label(r['Q']):>5} {r['omega_c']:>4} {r['lg_q']:>5} "
              f"{r['b_q_max']:>6} {r['ell']:>4} {r['p']:>3} {r['ell_prime']:>4} | "
              f"{_num_or_dash(r['lg_sigma_s'], 9)} {_num_or_dash(r['lg_sigma_R'], 9)} "
              f"{_num_or_dash(r['lg_sigma_r'], 9)} | {r['Sec_MQ']:>7.1f}")

def parameter_table_latex(dec=3):
    r"""Body rows for Table SEC-Selected-Parameters, in the column order of the paper:
    n, kappa, Q, omega_c, ceil(lg q), ell, p, ell', lg sigma_s, lg sigma_R, lg sigma_r."""
    fmt = "%." + str(dec) + "f"
    for i, g in enumerate(GEOMETRY):
        if i > 0:
            print(r"\midrule")
        for r in PARAMS:
            if r['n'] != g['n']:
                continue
            Qtex = "0" if r['Q'] == 0 else "2^{%d}" % (Integer(r['Q']).nbits() - 1)
            cells = [str(r['n']), str(r['kappa']), Qtex, str(r['omega_c']), str(r['lg_q']),
                     str(r['ell']), str(r['p']), str(r['ell_prime']),
                     fmt % float(r['lg_sigma_s']), fmt % float(r['lg_sigma_R'])]
            cells = [r"\(" + c + r"\)" for c in cells]
            cells.append("--" if r['lg_sigma_r'] is None else r"\(" + fmt % float(r['lg_sigma_r']) + r"\)")
            print(" & ".join(cells) + r" \\")

Printing.

In [92]:
parameter_table()
print()
parameter_table_latex()

  n  kap     Q  w_c  lg q  b_max  ell   p  l_p |  lg sig_s  lg sig_R  lg sig_r |  Sec_MQ
 40  128     0   19    32     35    6   3    3 |    2.9521    4.7312        -- |   150.4
 40  128  2^64   19    32     35    6   3    3 |    3.7238    5.2172   43.6412 |   150.4
 40  192     0   31    32     35    8   4    4 |    3.2323    4.9921        -- |   223.1
 40  192  2^64   31    32     35    8   4    4 |    4.0355    5.4748   44.6118 |   223.1
 40  256     0   44    32     35   10   5    5 |    3.4337    5.1830        -- |   299.6
 40  256  2^64   44    32     35   10   5    5 |    4.2540    5.6637   45.3101 |   299.6
 60  128     0   19    32     35    6   3    3 |    2.4579    5.2052        -- |   150.4
 60  128  2^64   19    32     35    6   3    3 |    2.6652    5.7063   43.9290 |   150.4
 60  192     0   31    32     35    8   4    4 |    2.7292    5.4645        -- |   223.1
 60  192  2^64   31    32     35    8   4    4 |    2.9406    5.9657   44.8946 |   223.1
 60  256     0   44  

# Computing the Sizes of the Secret and of One Share

## Secret and Share Sizes

Each participant stores exactly one share $\mathbf{s}_i \in \mathbb{Z}^{D}$, where $D := d\ell$. By $\mathbf{s}_i^\top = \mathbf{g}[i]\mathbf{s}^\top + \mathbf{\hat{G}}[i,\cdot]\mathbf{R}$, coefficient $j$ of $\mathbf{s}_i$ is $\mathbf{g}[i]\mathbf{s}[j] + \sum_{t=1}^{k-1}\mathbf{\hat{G}}[i,t]\mathbf{R}[t,j]$. For a fixed $i$, the coefficients are independent across $j$, and each has variance at most $\sigma_\mathsf{sh}^2 := \beta_\mathbf{g}^2\sigma_\mathbf{s}^2 + c_\mathrm{exp}\sigma_\mathbf{R}^2$, since every row of $\mathbf{\hat{G}}$ has exactly $c_\mathrm{exp}$ nonzero entries in $\{-1,1\}$.

As the ideal source model for a coefficient of standard deviation $\sigma$, we use a variance-matched discrete Gaussian, whose entropy is approximately $\lg\sigma + \frac{1}{2}\lg(2\pi e) \approx \lg\sigma + 2.047$ bits. Hence
\begin{align*}
    |\mathbf{s}| \approx D\left(\lg\sigma_\mathbf{s} + 2.047\right), \qquad |\mathbf{s}_i| \approx D\left(\lg\sigma_\mathsf{sh} + 2.047\right)
\end{align*}
bits. Unlike the $\mathsf{LaTSEC}$ notebook, $\sigma_\mathbf{s} \neq \sigma_\mathbf{R}$ in general, so we use $\sigma_\mathsf{sh}$ directly rather than the simplification $\sigma_\mathbf{s}\sqrt{1+c_\mathrm{exp}}$. These estimates exclude implementation-specific coding, framing, and byte-alignment overheads, and we use $1$ KiB $= 1024$ bytes.

In [93]:
### ---- secret and share sizes --------------------------------------------
# |s|   ~ D (lg sigma_s  + (1/2) lg(2 pi e)) bits
# |s_i| ~ D (lg sigma_sh + (1/2) lg(2 pi e)) bits,  sigma_sh^2 = beta_g^2 sigma_s^2 + c_exp sigma_R^2,
# with D = d ell; one share per participant.

def size_row(r):
    r"""Secret and share sizes (bits), lg sigma_sh, and lg beta_sh for one PARAMS row r."""
    key = (r['n'], r['kappa'], r['Q'], r['ell'])
    w, b = WIDTHS[key], NORM_BOUNDS[key]
    D = d * r['ell']
    return dict(secret_bits=size_bits(w['sigma_s'], D),
                share_bits=size_bits(w['sigma_sh'], D),
                lg_sigma_sh=lg(w['sigma_sh']),
                lg_beta_sh=lg(b['beta_sh']))

SIZES = {(r['n'], r['kappa'], r['Q']): size_row(r) for r in PARAMS}
assert_prec(SIZES, 'SIZES', ('secret_bits', 'share_bits', 'lg_sigma_sh', 'lg_beta_sh'))
for key, s in SIZES.items():
    assert s['share_bits'] > s['secret_bits'] > 0, key       # sigma_sh > sigma_s

## Size Table

For each $(n,\kappa,Q)$, the table reports the ideal entropy-coded sizes of the secret and of one share in KiB, their ratio, $\lg\sigma_\mathsf{sh}$, and the share bound $\lg\beta_\mathsf{sh}$. The LaTeX rows fill Table SEC-Selected-Sizes.

In [94]:
def size_table():
    r"""Secret and share sizes (KiB, 1 KiB = 1024 B), one row per (n, kappa, Q)."""
    KiB = lambda bits: float(bits) / (8*1024)
    print(f"{'n':>3} {'kap':>4} {'Q':>5} {'ell':>4} | {'secret KiB':>11} {'share KiB':>10} {'ratio':>6} | "
          f"{'lg sig_sh':>10} {'lg beta_sh':>11}")
    for r in PARAMS:
        s = SIZES[(r['n'], r['kappa'], r['Q'])]
        print(f"{r['n']:>3} {r['kappa']:>4} {Q_label(r['Q']):>5} {r['ell']:>4} | "
              f"{KiB(s['secret_bits']):>11.3f} {KiB(s['share_bits']):>10.3f} "
              f"{float(s['share_bits'] / s['secret_bits']):>6.3f} | "
              f"{float(s['lg_sigma_sh']):>10.4f} {float(s['lg_beta_sh']):>11.4f}")

Table of relevant size.

In [95]:
def size_table_latex(dec=3):
    r"""Body rows for Table SEC-Selected-Sizes: n, kappa, Q, secret (KiB), share (KiB)."""
    fmt = "%." + str(dec) + "f"
    KiB = lambda bits: float(bits) / (8*1024)
    for i, g in enumerate(GEOMETRY):
        if i > 0:
            print(r"\midrule")
        for r in PARAMS:
            if r['n'] != g['n']:
                continue
            s = SIZES[(r['n'], r['kappa'], r['Q'])]
            Qtex = "0" if r['Q'] == 0 else "2^{%d}" % (Integer(r['Q']).nbits() - 1)
            cells = [str(r['n']), str(r['kappa']), Qtex,
                     fmt % KiB(s['secret_bits']), fmt % KiB(s['share_bits'])]
            print(" & ".join(r"\(" + c + r"\)" for c in cells) + r" \\")

Printing the table.

In [96]:
size_table()
print()
size_table_latex()

  n  kap     Q  ell |  secret KiB  share KiB  ratio |  lg sig_sh  lg beta_sh
 40  128     0    6 |       1.875      2.847  1.518 |     5.5438     10.7628
 40  128  2^64    6 |       2.164      3.032  1.401 |     6.0393     11.3424
 40  192     0    8 |       2.640      3.926  1.487 |     5.8052     11.2734
 40  192  2^64    8 |       3.041      4.173  1.372 |     6.2992     11.8616
 40  256     0   10 |       3.425      5.027  1.468 |     5.9964     11.6465
 40  256  2^64   10 |       3.938      5.335  1.355 |     6.4895     12.2394
 60  128     0    6 |       1.689      3.019  1.787 |     6.0030     11.0066
 60  128  2^64    6 |       1.767      3.206  1.814 |     6.5024     11.4582
 60  192     0    8 |       2.388      4.155  1.740 |     6.2624     11.5143
 60  192  2^64    8 |       2.494      4.404  1.766 |     6.7618     11.9669
 60  256     0   10 |       3.108      5.312  1.709 |     6.4526     11.8855
 60  256  2^64   10 |       3.242      5.624  1.735 |     6.9520     12.3385